# Arabic + Coding AI Testing Sample Pipeline (Humanizer Disabled)

This notebook now uses the Arabic OCR workflow you built earlier:

1. Convert each Arabic PDF into a scanned image-based PDF.
2. OCR each scanned page with the OpenAI vision flow you used before.
3. Save raw OCR TXT files.
4. Trim the OCR text into AI-detection-ready TXT files.
5. Generate one prompt per trimmed Arabic file with the target word count matched to the trimmed file.
6. Generate one AI-written Arabic assignment per prompt.
7. Skip Arabic humanization for now.
8. Generate prompts for the coding TXT files.
9. Generate AI-written code files from those coding prompts.

The notebook is resume-friendly:
- skips files that already exist unless you turn overwrite on,
- writes progress CSVs,
- keeps one output file per source file.

In [ ]:
!pip install --quiet "openai>=1.0.0" pandas requests "pillow<12" "pymupdf>=1.24.0" "reportlab>=4.0.0"

import os
import re
import io
import gc
import csv
import time
import json
import math
import base64
import shutil
import requests
import pandas as pd
import fitz  # PyMuPDF
from pathlib import Path
from typing import List, Tuple
from PIL import Image
from openai import OpenAI
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader

from google.colab import drive

DRIVE_MOUNT_PATH = os.getenv("DRIVE_MOUNT_PATH", "/content/drive")
drive.mount(DRIVE_MOUNT_PATH, force_remount=False)

PROJECT_DATA_PATH = os.environ["PROJECT_DATA_PATH"]
BASE = Path(PROJECT_DATA_PATH).expanduser()

client = OpenAI()

HUMANIZER_EMAIL = os.getenv("HUMANIZER_EMAIL")
HUMANIZER_PASSWORD = os.getenv("HUMANIZER_PASSWORD")
HUMANIZER_API_URL = "https://ai-text-humanizer.com/api.php"

ARABIC_PDF_ROOT = BASE / "Arabic Assignments" / "2AI-Free Assignments"
ARABIC_SCANNED_PDF_ROOT = ARABIC_PDF_ROOT / "Scanned PDFs"
ARABIC_OCR_TXT_ROOT = ARABIC_PDF_ROOT / "OCR TXT"
ARABIC_TRIMMED_ROOT = ARABIC_PDF_ROOT / "Trimmed Assignments"
ARABIC_PROMPTS_ROOT = BASE / "Arabic Assignments" / "Prompts"
ARABIC_AI_ROOT = BASE / "Arabic Assignments" / "AI-Generated Sample"
ARABIC_HUMANIZED_ROOT = BASE / "Arabic Assignments" / "Humanized Sample"

CODING_FREE_ROOT = BASE / "Coding Assignments" / "AI-Free Assignments"
CODING_PROMPTS_ROOT = BASE / "Coding Assignments" / "Prompts"
CODING_AI_ROOT = BASE / "Coding Assignments" / "AI-Generated Sample"

PROGRESS_ROOT = BASE / "_pipeline_progress"
ARABIC_PROGRESS_CSV = PROGRESS_ROOT / "arabic_pipeline_progress.csv"
CODING_PROGRESS_CSV = PROGRESS_ROOT / "coding_pipeline_progress.csv"
ARABIC_TEMP_ROOT = BASE / "_tmp_arabic_pdf_ocr"

for path in [
    ARABIC_SCANNED_PDF_ROOT,
    ARABIC_OCR_TXT_ROOT,
    ARABIC_TRIMMED_ROOT,
    ARABIC_PROMPTS_ROOT,
    ARABIC_AI_ROOT,
    ARABIC_HUMANIZED_ROOT,
    CODING_PROMPTS_ROOT,
    CODING_AI_ROOT,
    PROGRESS_ROOT,
    ARABIC_TEMP_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

TRIM_MODEL = "gpt-4.1-mini"
PROMPT_MODEL = "gpt-4.1-mini"
GEN_MODEL = "gpt-4.1-mini"
OCR_MODEL = "gpt-4.1-mini"

MAX_CHARS_PER_TRIM_CHUNK = 6000
MAX_CHARS_FOR_PROMPT_INFERENCE = 45000
MAX_CHARS_FOR_CODE_PROMPT_INFERENCE = 60000
SLEEP_BETWEEN_OPENAI_CALLS = 1.0

OVERWRITE_EXISTING = False

# Rasterization limits control OCR request size and image quality.
OCR_RENDER_DPI = 170
OCR_JPEG_QUALITY = 85
OCR_MAX_IMAGE_DIM = 1800
OCR_MAX_RETRIES = 5
OCR_SLEEP_BETWEEN_PAGES = 0.4
OCR_REBUILD_SCANNED_IF_MISSING = True
OCR_REUSE_EXISTING_SCANNED_FOR_OCR = True
OCR_SKIP_IF_TXT_EXISTS = True
OCR_CLEAN_TEMP_AFTER_EACH_FILE = True

# Disabled because the external service did not produce suitable Arabic output.
RUN_ARABIC_HUMANIZER = False
MAX_WORDS_PER_HUMANIZER_REQUEST = 1500
HUMANIZER_CONNECT_TIMEOUT = 20
HUMANIZER_READ_TIMEOUT = 300
HUMANIZER_SLEEP_BETWEEN_REQUESTS = 2.0
HUMANIZER_RETRIES_PER_CHUNK = 3
HUMANIZER_BACKOFF_SECONDS = [10, 20, 40]

MIN_TRIMMED_WORDS_TO_KEEP = 120

print("Setup complete.")
print("Arabic PDF root:", ARABIC_PDF_ROOT)
print("Arabic scanned PDFs:", ARABIC_SCANNED_PDF_ROOT)
print("Arabic OCR text root:", ARABIC_OCR_TXT_ROOT)
print("Arabic trimmed root:", ARABIC_TRIMMED_ROOT)
print("Coding text root:", CODING_FREE_ROOT)
print("Arabic humanizer enabled:", RUN_ARABIC_HUMANIZER)

In [ ]:
# =========================
# 4) Shared helpers + Arabic OCR helpers
# =========================
def read_text_file(path: Path) -> str:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    raise ValueError(f"Could not read file: {path}")


def write_text_file(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text((text or "").strip() + "\n", encoding="utf-8")


def safe_stem(path: Path) -> str:
    stem = path.stem
    stem = re.sub(r"[^\w\-\.]+", "_", stem, flags=re.UNICODE).strip("._")
    return stem or "file"


def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text or ""))


def clean_model_output(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)

    intro_patterns = [
        r"^\s*here(?:'s| is)\b.*?\n+",
        r"^\s*certainly\b.*?\n+",
        r"^\s*sure\b.*?\n+",
        r"^\s*yes\b.*?\n+",
        r"^\s*i can help\b.*?\n+",
        r"^\s*below is\b.*?\n+",
        r"^\s*let(?: me)?(?:'s| us)?\b.*?\n+",
        r"^\s*of course\b.*?\n+",
        r"^\s*إليك\b.*?\n+",
        r"^\s*بالطبع\b.*?\n+",
        r"^\s*بالتأكيد\b.*?\n+",
    ]
    outro_patterns = [
        r"\n+\s*let me know if .*?$",
        r"\n+\s*if you(?:'d| would) like .*?$",
        r"\n+\s*feel free to .*?$",
        r"\n+\s*i can also .*?$",
        r"\n+\s*hope this helps.*?$",
        r"\n+\s*إذا رغبت .*?$",
        r"\n+\s*إذا أردت .*?$",
        r"\n+\s*أستطيع أيضًا .*?$",
    ]

    changed = True
    while changed:
        changed = False
        for pat in intro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    changed = True
    while changed:
        changed = False
        for pat in outro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    return text.strip()


def list_files(root: Path, exts: Tuple[str, ...]) -> List[Path]:
    return sorted(
        [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    )


def relative_txt_target(
    src_path: Path, src_root: Path, out_root: Path, suffix: str = ".txt"
) -> Path:
    rel = src_path.relative_to(src_root)
    parts = list(rel.parts)
    parts[-1] = safe_stem(Path(parts[-1])) + suffix
    return out_root.joinpath(*parts)


def relative_pdf_target(
    src_path: Path, src_root: Path, out_root: Path, suffix: str = ".pdf"
) -> Path:
    rel = src_path.relative_to(src_root)
    parts = list(rel.parts)
    parts[-1] = safe_stem(Path(parts[-1])) + suffix
    return out_root.joinpath(*parts)


def ensure_parent(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)


def load_progress_df(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            pass
    return pd.DataFrame()


def append_progress_row(csv_path: Path, row: dict):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def openai_response_text(messages, model: str) -> str:
    response = client.responses.create(model=model, input=messages)
    text = getattr(response, "output_text", "") or ""
    return text.strip()


def split_into_chunks_by_paragraphs(text: str, max_chars: int = 6000) -> List[str]:
    text = (text or "").replace("\r\n", "\n").replace("\r", "\n")
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        if not current:
            current = p
        elif len(current) + 2 + len(p) <= max_chars:
            current += "\n\n" + p
        else:
            chunks.append(current)
            current = p
    if current.strip():
        chunks.append(current)
    return chunks


# -------------------------
# Arabic OCR text helpers
# -------------------------
def normalize_text_basic(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def is_arabic_char(ch: str) -> bool:
    return (
        "\u0600" <= ch <= "\u06ff"
        or "\u0750" <= ch <= "\u077f"
        or "\u08a0" <= ch <= "\u08ff"
        or "\ufb50" <= ch <= "\ufdff"
        or "\ufe70" <= ch <= "\ufeff"
    )


def arabic_ratio(s: str) -> float:
    chars = [c for c in s if not c.isspace()]
    if not chars:
        return 0.0
    return sum(is_arabic_char(c) for c in chars) / len(chars)


def looks_like_bullet(line: str) -> bool:
    s = line.strip()
    return bool(
        re.match(
            r"^("
            r"[-*•●▪◦‣–—]\s+"
            r"|(?:\(?\d{1,3}\)?[\.\-:])\s+"
            r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:])\s+"
            r"|(?:\(?[A-Za-z]\)?[\.\-:])\s+"
            r"|(?:\(?[أ-ي]\)?[\.\-:])\s+"
            r")",
            s,
        )
    )


def looks_like_heading(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    if len(s) > 100:
        return False
    if looks_like_bullet(s):
        return False
    if s.endswith(":") or s.endswith("؛") or s.endswith("："):
        return True
    words = s.split()
    if 1 <= len(words) <= 12 and not re.search(r"[.!?؟]$", s):
        return arabic_ratio(s) >= 0.45
    return False


def should_preserve_newline_between(prev_line: str, next_line: str) -> bool:
    prev = prev_line.strip()
    nxt = next_line.strip()
    if not prev or not nxt:
        return True
    if looks_like_bullet(prev) or looks_like_bullet(nxt):
        return True
    if looks_like_heading(prev):
        return True
    if re.search(r"[.!?؟:؛]\s*$", prev):
        return True
    return False


def insert_newline_before_inline_bullets(text: str) -> str:
    pat = re.compile(
        r"(?<!\n)([^\n])\s+(?=("
        r"[-*•●▪◦‣–—]\s+"
        r"|(?:\(?\d{1,3}\)?[\.\-:]\s+)"
        r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:]\s+)"
        r"|(?:\(?[A-Za-z]\)?[\.\-:]\s+)"
        r"|(?:\(?[أ-ي]\)?[\.\-:]\s+)"
        r"))"
    )
    prev = None
    while prev != text:
        prev = text
        text = pat.sub(r"\1\n", text)
    return text


def merge_soft_linebreaks(text: str) -> str:
    text = normalize_text_basic(text)
    text = insert_newline_before_inline_bullets(text)
    lines = [ln.strip() for ln in text.split("\n")]
    out = []
    buf = []

    def flush_buf():
        nonlocal buf
        if buf:
            out.append(" ".join(buf).strip())
            buf = []

    for line in lines:
        if not line:
            flush_buf()
            continue
        if looks_like_bullet(line):
            flush_buf()
            out.append(line)
            continue
        if looks_like_heading(line):
            flush_buf()
            out.append(line)
            continue
        if not buf:
            buf = [line]
        else:
            prev_line = buf[-1]
            if should_preserve_newline_between(prev_line, line):
                flush_buf()
                buf = [line]
            else:
                buf.append(line)

    flush_buf()
    text = "\n".join(x for x in out if x.strip())
    text = re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"
    return text


def make_temp_dir_for(pdf_path: Path, source_root: Path, temp_root: Path) -> Path:
    rel = pdf_path.relative_to(source_root)
    return temp_root / rel.parent / safe_stem(rel)


def render_pdf_to_jpgs(pdf_path: Path, out_dir: Path, dpi: int = 170):
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(str(pdf_path))
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    page_paths = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img_path = out_dir / f"page_{i+1:04d}.jpg"
        pix.save(str(img_path))
        page_paths.append(img_path)
        del pix
        del page
        gc.collect()

    doc.close()
    return page_paths


def build_pdf_from_images(image_paths, out_pdf_path: Path):
    out_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    c = None
    try:
        first = True
        for img_path in image_paths:
            with Image.open(img_path) as im:
                width_px, height_px = im.size
                page_w = float(width_px)
                page_h = float(height_px)

            if first:
                c = canvas.Canvas(str(out_pdf_path), pagesize=(page_w, page_h))
                first = False
            else:
                c.setPageSize((page_w, page_h))

            c.drawImage(ImageReader(str(img_path)), 0, 0, width=page_w, height=page_h)
            c.showPage()

        if c is None:
            raise RuntimeError("No images provided to build scanned PDF.")
        c.save()
    finally:
        if c is not None:
            try:
                del c
            except Exception:
                pass


def image_to_data_url(img_path: Path) -> str:
    with Image.open(img_path) as im:
        im = im.convert("RGB")
        w, h = im.size
        scale = min(1.0, OCR_MAX_IMAGE_DIM / max(w, h))
        if scale < 1.0:
            im = im.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=OCR_JPEG_QUALITY, optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


def openai_ocr_image(img_path: Path) -> str:
    data_url = image_to_data_url(img_path)
    prompt = (
        "Transcribe this page exactly into plain text.\n"
        "Rules:\n"
        "1) Preserve Arabic text exactly as written.\n"
        "2) Also preserve English text, numbers, and visible symbols.\n"
        "3) Do not summarize.\n"
        "4) Do not explain.\n"
        "5) Do not correct wording.\n"
        "6) If a line break exists only because the printed line reached page width, do NOT keep it.\n"
        "7) Keep true paragraph breaks, headings, and bullet/numbered-point breaks.\n"
        "8) Return only the page text."
    )

    last_err = None
    for attempt in range(1, OCR_MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=OCR_MODEL,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": prompt},
                            {"type": "input_image", "image_url": data_url},
                        ],
                    }
                ],
            )
            txt = (resp.output_text or "").strip()
            return txt
        except Exception as e:
            last_err = e
            wait_s = min(20, 2 ** (attempt - 1))
            print(
                f"    OpenAI OCR retry {attempt}/{OCR_MAX_RETRIES} for {img_path.name}: {e}"
            )
            time.sleep(wait_s)

    raise RuntimeError(f"OpenAI OCR failed for {img_path.name}: {last_err}")


def ocr_images_to_txt(image_paths, txt_out_path: Path):
    page_texts = []
    for img_path in image_paths:
        txt = openai_ocr_image(img_path)
        txt = normalize_text_basic(txt)
        txt = merge_soft_linebreaks(txt)
        page_texts.append(txt.strip())
        time.sleep(OCR_SLEEP_BETWEEN_PAGES)

    final_text = "\n\n".join(t for t in page_texts if t.strip()).strip() + "\n"
    txt_out_path.parent.mkdir(parents=True, exist_ok=True)
    txt_out_path.write_text(final_text, encoding="utf-8")


def build_scanned_pdf_and_ocr_txt(pdf_path: Path) -> Tuple[Path, Path]:
    scanned_pdf_path = relative_pdf_target(
        pdf_path, ARABIC_PDF_ROOT, ARABIC_SCANNED_PDF_ROOT
    )
    ocr_txt_path = relative_txt_target(pdf_path, ARABIC_PDF_ROOT, ARABIC_OCR_TXT_ROOT)

    if OCR_SKIP_IF_TXT_EXISTS and ocr_txt_path.exists() and not OVERWRITE_EXISTING:
        return scanned_pdf_path, ocr_txt_path

    temp_dir = make_temp_dir_for(pdf_path, ARABIC_PDF_ROOT, ARABIC_TEMP_ROOT)
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        image_paths = render_pdf_to_jpgs(pdf_path, temp_dir, dpi=OCR_RENDER_DPI)

        if (
            OVERWRITE_EXISTING
            or not scanned_pdf_path.exists()
            or OCR_REBUILD_SCANNED_IF_MISSING
        ):
            build_pdf_from_images(image_paths, scanned_pdf_path)

        if OVERWRITE_EXISTING or not ocr_txt_path.exists():
            ocr_images_to_txt(image_paths, ocr_txt_path)

        return scanned_pdf_path, ocr_txt_path
    finally:
        if OCR_CLEAN_TEMP_AFTER_EACH_FILE and temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)

In [ ]:
ARABIC_TRIM_SYSTEM = """
Clean extracted Arabic student-authored assignment text for preprocessing.

Do not rewrite, paraphrase, summarize, translate, reorder, or add words. Only delete text and join clearly broken lines within a paragraph.

Keep complete student-authored Arabic prose, coherent answer lists, and relevant section titles.

Delete course and cover-page metadata, names and IDs, dates, signatures, repeated headers and footers, tables, captions, references, appendices, page numbers, form fields, template instructions, equation- or symbol-heavy text, OCR corruption, repeated garbage, and isolated fragments.

Preserve the original wording of retained text. Return only the cleaned text.
"""

ARABIC_PROMPT_SYSTEM = """
Infer one realistic Arabic prompt a student could have used to request an assignment similar in topic, deliverable, structure, tone, course level, and scope to the supplied trimmed text.

The prompt must:
- sound like a real student request in Arabic;
- ask for writing or completing the assignment, not analyzing it;
- explicitly request an approximate final word count equal to the supplied target;
- avoid quoting long passages from the source;
- not mention AI detection, reverse engineering, evaluation, or research.

Return only the prompt text.
"""

ARABIC_GENERATION_SYSTEM = """
Write only the requested Arabic student assignment.
Do not add prefatory text, comments, disclaimers, notes, or references to AI.
"""


def trim_arabic_text_with_openai(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(
        raw_text,
        max_chars=MAX_CHARS_PER_TRIM_CHUNK,
    )
    cleaned_chunks = []

    for index, chunk in enumerate(chunks, start=1):
        messages = [
            {"role": "system", "content": ARABIC_TRIM_SYSTEM.strip()},
            {
                "role": "user",
                "content": (
                    f"هذا هو الجزء رقم {index} من النص المستخرج. "
                    f"نظّفه حسب التعليمات وأعد فقط النص المحتفَظ به:\n\n{chunk}"
                ),
            },
        ]
        output = clean_model_output(openai_response_text(messages, model=TRIM_MODEL))
        if output.strip():
            cleaned_chunks.append(output.strip())
        time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)

    merged = "\n\n".join(cleaned_chunks).strip()

    # A final pass can reconnect paragraph lines split across chunk boundaries.
    if merged:
        second_pass_system = (
            ARABIC_TRIM_SYSTEM
            + "\nJoin clearly broken lines within the same paragraph without rewriting."
        )
        messages = [
            {"role": "system", "content": second_pass_system.strip()},
            {
                "role": "user",
                "content": (
                    "نفّذ تنظيفًا نهائيًا على النص التالي وأعد فقط النسخة "
                    f"النهائية المحتفَظ بها:\n\n{merged[:120000]}"
                ),
            },
        ]
        merged = clean_model_output(openai_response_text(messages, model=TRIM_MODEL))
        time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)

    return merged.strip()


def build_arabic_prompt(
    file_name: str,
    trimmed_text: str,
    target_word_count: int,
) -> str:
    truncated_text = trimmed_text[:MAX_CHARS_FOR_PROMPT_INFERENCE]
    messages = [
        {"role": "system", "content": ARABIC_PROMPT_SYSTEM.strip()},
        {
            "role": "user",
            "content": f"""اسم الملف: {file_name}
عدد الكلمات المطلوب ذكره في الطلب: حوالي {target_word_count} كلمة

نص الواجب العربي النهائي:
--------------------
{truncated_text}
--------------------

اكتب أفضل طلب واحد فقط بصياغة طالب حقيقية وباللغة العربية.""",
        },
    ]
    prompt_text = clean_model_output(openai_response_text(messages, model=PROMPT_MODEL))
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return prompt_text


def generate_arabic_assignment(prompt_text: str) -> str:
    messages = [
        {"role": "system", "content": ARABIC_GENERATION_SYSTEM.strip()},
        {"role": "user", "content": prompt_text},
    ]
    text = clean_model_output(openai_response_text(messages, model=GEN_MODEL))
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return text


def split_into_paragraphs(text: str) -> List[str]:
    text = (text or "").replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return []
    return [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n+", text)
        if paragraph.strip()
    ]


def split_long_paragraph(paragraph: str, max_words: int) -> List[str]:
    words = re.findall(r"\S+", paragraph)
    if len(words) <= max_words:
        return [paragraph]

    sentences = re.split(r"(?<=[.!؟?])\s+", paragraph.strip())
    chunks: List[str] = []
    current: List[str] = []

    def current_word_count() -> int:
        return count_words(" ".join(current)) if current else 0

    for sentence in sentences:
        sentence_word_count = count_words(sentence)
        if sentence_word_count > max_words:
            sentence_words = re.findall(r"\S+", sentence)
            if current:
                chunks.append(" ".join(current).strip())
                current = []
            for start in range(0, len(sentence_words), max_words):
                part = " ".join(sentence_words[start : start + max_words]).strip()
                if part:
                    chunks.append(part)
            continue

        if current_word_count() + sentence_word_count <= max_words:
            current.append(sentence)
        else:
            if current:
                chunks.append(" ".join(current).strip())
            current = [sentence]

    if current:
        chunks.append(" ".join(current).strip())

    return [chunk for chunk in chunks if chunk.strip()]


def chunk_text_by_paragraphs(text: str, max_words: int = 1500) -> List[str]:
    paragraphs = split_into_paragraphs(text)
    if not paragraphs:
        return []

    chunks: List[str] = []
    current_parts: List[str] = []
    current_word_count = 0

    for paragraph in paragraphs:
        paragraph_word_count = count_words(paragraph)

        if paragraph_word_count > max_words:
            if current_parts:
                chunks.append("\n\n".join(current_parts).strip())
                current_parts = []
                current_word_count = 0
            chunks.extend(split_long_paragraph(paragraph, max_words))
            continue

        if current_word_count + paragraph_word_count <= max_words:
            current_parts.append(paragraph)
            current_word_count += paragraph_word_count
        else:
            chunks.append("\n\n".join(current_parts).strip())
            current_parts = [paragraph]
            current_word_count = paragraph_word_count

    if current_parts:
        chunks.append("\n\n".join(current_parts).strip())

    return chunks


def call_humanizer_api(text: str) -> str:
    data = {
        "email": HUMANIZER_EMAIL,
        "pw": HUMANIZER_PASSWORD,
        "text": text,
    }
    response = requests.post(
        HUMANIZER_API_URL,
        data=data,
        timeout=(HUMANIZER_CONNECT_TIMEOUT, HUMANIZER_READ_TIMEOUT),
    )
    response.raise_for_status()
    output = response.text.strip()

    try:
        parsed = response.json()
    except ValueError:
        parsed = None

    if isinstance(parsed, dict):
        for key in ("output", "text", "humanized_text", "result", "message"):
            value = parsed.get(key)
            if isinstance(value, str) and value.strip():
                output = value.strip()
                break

    return output.strip()


def humanize_text(text: str) -> str:
    chunks = chunk_text_by_paragraphs(
        text,
        max_words=MAX_WORDS_PER_HUMANIZER_REQUEST,
    )
    outputs = []

    for index, chunk in enumerate(chunks, start=1):
        success = False
        last_error = None

        for attempt in range(HUMANIZER_RETRIES_PER_CHUNK):
            try:
                output = clean_model_output(call_humanizer_api(chunk))
                if output.strip():
                    outputs.append(output.strip())
                    success = True
                    break
                raise ValueError("Empty humanizer output.")
            except Exception as error:
                last_error = error
                if attempt < len(HUMANIZER_BACKOFF_SECONDS):
                    time.sleep(HUMANIZER_BACKOFF_SECONDS[attempt])

        if not success:
            raise RuntimeError(f"Humanizer failed on chunk {index}: {last_error}")

        time.sleep(HUMANIZER_SLEEP_BETWEEN_REQUESTS)

    return "\n\n".join(outputs).strip()


def run_arabic_pipeline():
    pdf_files = list_files(ARABIC_PDF_ROOT, (".pdf",))
    if not pdf_files:
        raise FileNotFoundError(f"No Arabic PDF files found under: {ARABIC_PDF_ROOT}")

    print(f"Found {len(pdf_files)} Arabic PDFs.")

    for index, pdf_path in enumerate(pdf_files, start=1):
        if ARABIC_SCANNED_PDF_ROOT in pdf_path.parents:
            continue

        scanned_pdf_path = relative_pdf_target(
            pdf_path,
            ARABIC_PDF_ROOT,
            ARABIC_SCANNED_PDF_ROOT,
        )
        ocr_txt_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_OCR_TXT_ROOT
        )
        trimmed_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_TRIMMED_ROOT
        )
        prompt_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_PROMPTS_ROOT
        )
        ai_path = relative_txt_target(pdf_path, ARABIC_PDF_ROOT, ARABIC_AI_ROOT)
        humanized_path = relative_txt_target(
            pdf_path,
            ARABIC_PDF_ROOT,
            ARABIC_HUMANIZED_ROOT,
        )

        status_row = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source_pdf": str(pdf_path),
            "scanned_pdf": str(scanned_pdf_path),
            "ocr_txt": str(ocr_txt_path),
            "trimmed_txt": str(trimmed_path),
            "prompt_txt": str(prompt_path),
            "ai_txt": str(ai_path),
            "humanized_txt": str(humanized_path),
            "ocr_words": 0,
            "trim_words": 0,
            "ai_words": 0,
            "humanized_words": 0,
            "status": "",
            "error": "",
        }

        print(f"[Arabic {index}/{len(pdf_files)}] {pdf_path.name}")

        try:
            if (
                OVERWRITE_EXISTING
                or not ocr_txt_path.exists()
                or not scanned_pdf_path.exists()
            ):
                scanned_pdf_path, ocr_txt_path = build_scanned_pdf_and_ocr_txt(pdf_path)
            raw_text = read_text_file(ocr_txt_path)
            status_row["ocr_words"] = count_words(raw_text)

            if OVERWRITE_EXISTING or not trimmed_path.exists():
                trimmed = trim_arabic_text_with_openai(raw_text)
                if count_words(trimmed) < MIN_TRIMMED_WORDS_TO_KEEP:
                    raise ValueError(
                        f"Trimmed output too small: {count_words(trimmed)} words"
                    )
                write_text_file(trimmed_path, trimmed)
            else:
                trimmed = read_text_file(trimmed_path)

            trim_word_count = count_words(trimmed)
            status_row["trim_words"] = trim_word_count

            if OVERWRITE_EXISTING or not prompt_path.exists():
                prompt_text = build_arabic_prompt(
                    pdf_path.name,
                    trimmed,
                    trim_word_count,
                )
                write_text_file(prompt_path, prompt_text)
            else:
                prompt_text = read_text_file(prompt_path)

            if OVERWRITE_EXISTING or not ai_path.exists():
                ai_text = generate_arabic_assignment(prompt_text)
                write_text_file(ai_path, ai_text)
            else:
                ai_text = read_text_file(ai_path)

            status_row["ai_words"] = count_words(ai_text)

            if RUN_ARABIC_HUMANIZER:
                if OVERWRITE_EXISTING or not humanized_path.exists():
                    humanized_text = humanize_text(ai_text)
                    write_text_file(humanized_path, humanized_text)
                else:
                    humanized_text = read_text_file(humanized_path)
                status_row["humanized_words"] = count_words(humanized_text)
            else:
                status_row["humanized_words"] = 0

            status_row["status"] = "ok"
            append_progress_row(ARABIC_PROGRESS_CSV, status_row)

            if RUN_ARABIC_HUMANIZER:
                print(
                    f"  -> ok | ocr={status_row['ocr_words']} "
                    f"| trim={trim_word_count} "
                    f"| ai={status_row['ai_words']} "
                    f"| humanized={status_row['humanized_words']}"
                )
            else:
                print(
                    f"  -> ok | ocr={status_row['ocr_words']} "
                    f"| trim={trim_word_count} "
                    f"| ai={status_row['ai_words']} | humanizer=skipped"
                )

        except Exception as error:
            status_row["status"] = "failed"
            status_row["error"] = str(error)
            append_progress_row(ARABIC_PROGRESS_CSV, status_row)
            print(f"  -> FAILED: {error}")


print("Arabic pipeline functions ready.")

In [ ]:
# =========================
# 6) Coding prompts + AI code generation
# =========================
CODE_PROMPT_SYSTEM = """
You are helping with a research workflow.

Task:
Read a student's code sample taken from high-signal parts of a real repository and infer ONE realistic prompt that the student could have asked ChatGPT to produce a similar coding submission.

Requirements:
1. The prompt must sound like a real student request.
2. It must ask for writing/building the code itself, not analyzing it.
3. It must capture the likely project/task, language, features, structure, and difficulty level.
4. It should request code similar in scope and style to the sample.
5. It MUST explicitly request approximately the same total length as the sample.
6. If the sample clearly uses a specific language/framework, preserve that in the prompt.
7. Do NOT quote long passages from the sample.
8. Do NOT mention AI detection, reverse engineering, evaluation, or research.
9. Output ONLY the prompt text.
"""

CODE_GENERATION_SYSTEM = """
You are writing the final code deliverable only.

Return only the requested code/content itself.
Do NOT add explanations before or after.
Do NOT add markdown code fences unless the prompt itself explicitly requires them.
Do NOT add notes like 'Here is the code'.
Output only the final code text.
"""


def build_code_prompt(file_name: str, code_text: str, word_count: int) -> str:
    truncated = code_text[:MAX_CHARS_FOR_CODE_PROMPT_INFERENCE]
    messages = [
        {"role": "system", "content": CODE_PROMPT_SYSTEM.strip()},
        {
            "role": "user",
            "content": f"""File name: {file_name}
Approximate target length to request: about {word_count} words/tokens of code text

Code sample:
--------------------
{truncated}
--------------------

Write the single best student-style prompt.""",
        },
    ]
    out = clean_model_output(openai_response_text(messages, model=PROMPT_MODEL))
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return out


def generate_code_from_prompt(prompt_text: str) -> str:
    messages = [
        {"role": "system", "content": CODE_GENERATION_SYSTEM.strip()},
        {"role": "user", "content": prompt_text},
    ]
    out = clean_model_output(openai_response_text(messages, model=GEN_MODEL))
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return out


def run_coding_pipeline():
    txt_files = list_files(CODING_FREE_ROOT, (".txt",))
    if not txt_files:
        raise FileNotFoundError(f"No coding TXT files found under: {CODING_FREE_ROOT}")

    print(f"Found {len(txt_files)} coding TXT files.")

    for i, txt_path in enumerate(txt_files, start=1):
        prompt_path = relative_txt_target(
            txt_path, CODING_FREE_ROOT, CODING_PROMPTS_ROOT
        )
        ai_path = relative_txt_target(txt_path, CODING_FREE_ROOT, CODING_AI_ROOT)

        status_row = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source_txt": str(txt_path),
            "prompt_txt": str(prompt_path),
            "ai_txt": str(ai_path),
            "source_words": 0,
            "ai_words": 0,
            "status": "",
            "error": "",
        }

        print(f"[Coding {i}/{len(txt_files)}] {txt_path.name}")

        try:
            code_text = read_text_file(txt_path)
            source_wc = count_words(code_text)
            status_row["source_words"] = source_wc

            if OVERWRITE_EXISTING or not prompt_path.exists():
                prompt_text = build_code_prompt(txt_path.name, code_text, source_wc)
                write_text_file(prompt_path, prompt_text)
            else:
                prompt_text = read_text_file(prompt_path)

            if OVERWRITE_EXISTING or not ai_path.exists():
                ai_text = generate_code_from_prompt(prompt_text)
                write_text_file(ai_path, ai_text)
            else:
                ai_text = read_text_file(ai_path)

            status_row["ai_words"] = count_words(ai_text)
            status_row["status"] = "ok"
            append_progress_row(CODING_PROGRESS_CSV, status_row)
            print(f"  -> ok | source={source_wc} | ai={status_row['ai_words']}")

        except Exception as e:
            status_row["status"] = "failed"
            status_row["error"] = str(e)
            append_progress_row(CODING_PROGRESS_CSV, status_row)
            print(f"  -> FAILED: {e}")


print("Coding pipeline functions ready.")

In [ ]:
# Quota-aware, resumable Arabic OCR
ARABIC_OCR_PROGRESS_JSON = PROGRESS_ROOT / "arabic_ocr_progress.json"
ARABIC_OCR_PENDING_TXT = PROGRESS_ROOT / "arabic_ocr_pending_files.txt"
ARABIC_OCR_FAILED_TXT = PROGRESS_ROOT / "arabic_ocr_failed_files.txt"
OCR_STOP_ON_INSUFFICIENT_QUOTA = True
OCR_PAGE_MARKER_TEMPLATE = "############ PAGE {page_no} ############"


def load_json_safe(path: Path, default):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    return default


def save_json_safe(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def append_line(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as file:
        file.write((text or "").rstrip() + "\n")


def clear_file(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("", encoding="utf-8")


def is_insufficient_quota_error(exc: Exception) -> bool:
    message = str(exc).lower()
    return "insufficient_quota" in message or "exceeded your current quota" in message


def is_temporary_rate_limit_error(exc: Exception) -> bool:
    message = str(exc).lower()
    return (
        "rate limit" in message or "too many requests" in message
    ) and not is_insufficient_quota_error(exc)


def openai_ocr_image(img_path: Path) -> str:
    data_url = image_to_data_url(img_path)
    prompt = (
        "Transcribe this page exactly into plain text.\n"
        "Rules:\n"
        "1) Preserve Arabic text exactly as written.\n"
        "2) Also preserve English text, numbers, and visible symbols.\n"
        "3) Do not summarize.\n"
        "4) Do not explain.\n"
        "5) Do not correct wording.\n"
        "6) If a line break exists only because the printed line reached page width, do NOT keep it.\n"
        "7) Keep true paragraph breaks, headings, and bullet/numbered-point breaks.\n"
        "8) Return only the page text."
    )

    last_error = None
    for attempt in range(1, OCR_MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=OCR_MODEL,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": prompt},
                            {"type": "input_image", "image_url": data_url},
                        ],
                    }
                ],
            )
            text = (getattr(response, "output_text", "") or "").strip()
            if not text:
                raise RuntimeError(f"Empty OCR response for {img_path.name}")
            return text
        except Exception as exc:
            last_error = exc

            if OCR_STOP_ON_INSUFFICIENT_QUOTA and is_insufficient_quota_error(exc):
                raise RuntimeError(f"INSUFFICIENT_QUOTA::{img_path.name}::{exc}")

            if attempt < OCR_MAX_RETRIES:
                wait_seconds = (
                    min(60, 5 * attempt)
                    if is_temporary_rate_limit_error(exc)
                    else min(20, 2 ** (attempt - 1))
                )
                print(
                    f"    OpenAI OCR retry {attempt}/{OCR_MAX_RETRIES} "
                    f"for {img_path.name}: {exc}"
                )
                time.sleep(wait_seconds)
                continue
            break

    raise RuntimeError(f"OpenAI OCR failed for {img_path.name}: {last_error}")


def parse_checkpointed_ocr_txt(txt_path: Path) -> dict:
    if not txt_path.exists():
        return {}

    raw_text = read_text_file(txt_path)
    if not raw_text.strip():
        return {}

    pages = {}
    pattern = re.compile(
        r"^############ PAGE\s+(\d+)\s+############\s*$",
        flags=re.MULTILINE,
    )
    matches = list(pattern.finditer(raw_text))
    if not matches:
        return {}

    for index, match in enumerate(matches):
        page_no = int(match.group(1))
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(raw_text)
        body = raw_text[start:end].strip()
        if body:
            pages[page_no] = body

    return pages


def write_checkpointed_ocr_txt(txt_out_path: Path, page_texts: dict):
    txt_out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(txt_out_path, "w", encoding="utf-8") as file:
        for page_no in sorted(page_texts):
            file.write(OCR_PAGE_MARKER_TEMPLATE.format(page_no=page_no) + "\n")
            file.write((page_texts[page_no] or "").rstrip() + "\n\n")


def ocr_images_to_txt(image_paths, txt_out_path: Path, progress_key: str = None):
    progress = load_json_safe(ARABIC_OCR_PROGRESS_JSON, {})
    progress_key = progress_key or txt_out_path.stem
    state = progress.setdefault(
        progress_key,
        {
            "done": False,
            "fatal_quota": False,
            "page_count": len(image_paths),
            "pages_completed": [],
            "txt_out_path": str(txt_out_path),
        },
    )

    recovered_pages = parse_checkpointed_ocr_txt(txt_out_path)
    completed_pages = set(state.get("pages_completed", [])) | set(recovered_pages)

    for index, img_path in enumerate(image_paths, start=1):
        # Require both progress metadata and page text before skipping a page.
        if index in completed_pages and index in recovered_pages:
            continue

        try:
            text = openai_ocr_image(img_path)
        except Exception as exc:
            if str(exc).startswith("INSUFFICIENT_QUOTA::"):
                state["fatal_quota"] = True
                state["done"] = False
                state["pages_completed"] = sorted(completed_pages)
                save_json_safe(ARABIC_OCR_PROGRESS_JSON, progress)
                append_line(ARABIC_OCR_PENDING_TXT, progress_key)
                raise RuntimeError(str(exc))
            raise

        text = merge_soft_linebreaks(normalize_text_basic(text))
        recovered_pages[index] = text.strip()
        completed_pages.add(index)

        state["page_count"] = len(image_paths)
        state["pages_completed"] = sorted(completed_pages)
        state["fatal_quota"] = False
        write_checkpointed_ocr_txt(txt_out_path, recovered_pages)
        save_json_safe(ARABIC_OCR_PROGRESS_JSON, progress)
        time.sleep(OCR_SLEEP_BETWEEN_PAGES)

    state["done"] = True
    state["fatal_quota"] = False
    state["pages_completed"] = sorted(completed_pages)
    save_json_safe(ARABIC_OCR_PROGRESS_JSON, progress)


def build_scanned_pdf_and_ocr_txt(pdf_path: Path) -> Tuple[Path, Path]:
    scanned_pdf_path = relative_pdf_target(
        pdf_path,
        ARABIC_PDF_ROOT,
        ARABIC_SCANNED_PDF_ROOT,
    )
    ocr_txt_path = relative_txt_target(
        pdf_path,
        ARABIC_PDF_ROOT,
        ARABIC_OCR_TXT_ROOT,
    )
    progress_key = str(pdf_path.relative_to(ARABIC_PDF_ROOT)).replace("\\", "/")

    progress = load_json_safe(ARABIC_OCR_PROGRESS_JSON, {})
    state = progress.get(progress_key, {})
    if (
        OCR_SKIP_IF_TXT_EXISTS
        and ocr_txt_path.exists()
        and state.get("done")
        and not OVERWRITE_EXISTING
    ):
        return scanned_pdf_path, ocr_txt_path

    temp_dir = make_temp_dir_for(pdf_path, ARABIC_PDF_ROOT, ARABIC_TEMP_ROOT)
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        image_paths = render_pdf_to_jpgs(pdf_path, temp_dir, dpi=OCR_RENDER_DPI)

        if (
            OVERWRITE_EXISTING
            or not scanned_pdf_path.exists()
            or OCR_REBUILD_SCANNED_IF_MISSING
        ):
            build_pdf_from_images(image_paths, scanned_pdf_path)

        if OVERWRITE_EXISTING or not ocr_txt_path.exists() or not state.get("done"):
            ocr_images_to_txt(image_paths, ocr_txt_path, progress_key=progress_key)

        return scanned_pdf_path, ocr_txt_path
    finally:
        if OCR_CLEAN_TEMP_AFTER_EACH_FILE and temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)


def run_arabic_pipeline():
    pdf_files = [
        path
        for path in list_files(ARABIC_PDF_ROOT, (".pdf",))
        if ARABIC_SCANNED_PDF_ROOT not in path.parents
    ]
    if not pdf_files:
        raise FileNotFoundError(f"No Arabic PDF files found under: {ARABIC_PDF_ROOT}")

    clear_file(ARABIC_OCR_PENDING_TXT)
    print(f"Found {len(pdf_files)} Arabic PDFs.")

    stop_all = False
    for index, pdf_path in enumerate(pdf_files, start=1):
        if stop_all:
            break

        scanned_pdf_path = relative_pdf_target(
            pdf_path,
            ARABIC_PDF_ROOT,
            ARABIC_SCANNED_PDF_ROOT,
        )
        ocr_txt_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_OCR_TXT_ROOT
        )
        trimmed_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_TRIMMED_ROOT
        )
        prompt_path = relative_txt_target(
            pdf_path, ARABIC_PDF_ROOT, ARABIC_PROMPTS_ROOT
        )
        ai_path = relative_txt_target(pdf_path, ARABIC_PDF_ROOT, ARABIC_AI_ROOT)
        humanized_path = relative_txt_target(
            pdf_path,
            ARABIC_PDF_ROOT,
            ARABIC_HUMANIZED_ROOT,
        )

        status_row = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source_pdf": str(pdf_path),
            "scanned_pdf": str(scanned_pdf_path),
            "ocr_txt": str(ocr_txt_path),
            "trimmed_txt": str(trimmed_path),
            "prompt_txt": str(prompt_path),
            "ai_txt": str(ai_path),
            "humanized_txt": str(humanized_path),
            "ocr_words": 0,
            "trim_words": 0,
            "ai_words": 0,
            "humanized_words": 0,
            "status": "",
            "error": "",
        }

        print(f"[Arabic {index}/{len(pdf_files)}] {pdf_path.name}")

        try:
            if (
                OVERWRITE_EXISTING
                or not ocr_txt_path.exists()
                or not scanned_pdf_path.exists()
            ):
                scanned_pdf_path, ocr_txt_path = build_scanned_pdf_and_ocr_txt(pdf_path)

            raw_text = read_text_file(ocr_txt_path)
            status_row["ocr_words"] = count_words(raw_text)

            if OVERWRITE_EXISTING or not trimmed_path.exists():
                trimmed = trim_arabic_text_with_openai(raw_text)
                if count_words(trimmed) < MIN_TRIMMED_WORDS_TO_KEEP:
                    raise ValueError(
                        f"Trimmed output too small: {count_words(trimmed)} words"
                    )
                write_text_file(trimmed_path, trimmed)
            else:
                trimmed = read_text_file(trimmed_path)

            trim_word_count = count_words(trimmed)
            status_row["trim_words"] = trim_word_count

            if OVERWRITE_EXISTING or not prompt_path.exists():
                prompt_text = build_arabic_prompt(
                    pdf_path.name, trimmed, trim_word_count
                )
                write_text_file(prompt_path, prompt_text)
            else:
                prompt_text = read_text_file(prompt_path)

            if OVERWRITE_EXISTING or not ai_path.exists():
                ai_text = generate_arabic_assignment(prompt_text)
                write_text_file(ai_path, ai_text)
            else:
                ai_text = read_text_file(ai_path)

            status_row["ai_words"] = count_words(ai_text)

            if OVERWRITE_EXISTING or not humanized_path.exists():
                humanized_text = " "
            else:
                humanized_text = read_text_file(humanized_path)

            status_row["humanized_words"] = count_words(humanized_text)
            status_row["status"] = "ok"
            append_progress_row(ARABIC_PROGRESS_CSV, status_row)
            print(
                f"  -> ok | ocr={status_row['ocr_words']} | trim={trim_word_count} "
                f"| ai={status_row['ai_words']} | humanized={status_row['humanized_words']}"
            )

        except Exception as exc:
            status_row["status"] = "failed"
            status_row["error"] = str(exc)
            append_progress_row(ARABIC_PROGRESS_CSV, status_row)
            append_line(ARABIC_OCR_FAILED_TXT, f"{pdf_path.name}\t{exc}")

            if "INSUFFICIENT_QUOTA::" in str(exc):
                print(
                    "  -> STOPPED: OpenAI quota exhausted. Progress was saved. Resume later after quota/billing is fixed."
                )
                stop_all = True
            else:
                print(f"  -> FAILED: {exc}")

    print("Arabic pipeline functions ready. Quota-safe OCR patch active.")

In [ ]:
# =========================
# 7) Run
# =========================
# Run these one by one if you want, or run both.
# Arabic humanizer stage is disabled in this version.

run_arabic_pipeline()
run_coding_pipeline()

print("\nDone.")
print("Arabic scanned  ->", ARABIC_SCANNED_PDF_ROOT)
print("Arabic OCR TXT  ->", ARABIC_OCR_TXT_ROOT)
print("Arabic trimmed  ->", ARABIC_TRIMMED_ROOT)
print("Arabic prompts  ->", ARABIC_PROMPTS_ROOT)
print("Arabic AI       ->", ARABIC_AI_ROOT)
print(
    "Arabic humanized->",
    (
        f"{ARABIC_HUMANIZED_ROOT} (skipped)"
        if not RUN_ARABIC_HUMANIZER
        else ARABIC_HUMANIZED_ROOT
    ),
)
print("Coding prompts  ->", CODING_PROMPTS_ROOT)
print("Coding AI       ->", CODING_AI_ROOT)
print("Arabic progress ->", ARABIC_PROGRESS_CSV)
print("Arabic OCR JSON ->", ARABIC_OCR_PROGRESS_JSON)
print("Arabic OCR pending ->", ARABIC_OCR_PENDING_TXT)
print("Arabic OCR failed  ->", ARABIC_OCR_FAILED_TXT)
print("Coding progress ->", CODING_PROGRESS_CSV)

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

project_data_path = os.environ.get("PROJECT_DATA_PATH")
if not project_data_path:
    raise ValueError("Set PROJECT_DATA_PATH to the project data directory.")

BASE = Path(project_data_path).expanduser()
FOLDERS = {
    "Arabic_Trimmed_AI_Free": BASE
    / "Arabic Assignments"
    / "Trimmed AI-Free Assignments",
    "Arabic_AI_Generated": BASE / "Arabic Assignments" / "AI-Generated Assignments",
    "Coding_AI_Free": BASE / "Coding Assignments" / "AI-Free Assignments",
    "Coding_AI_Generated": BASE / "Coding Assignments" / "AI-Generated Code",
}

REPORT_DIR = BASE / "_word_count_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BIN_SIZE = 1000
ARABIC_FOLDER_KEYS = {"Arabic_Trimmed_AI_Free", "Arabic_AI_Generated"}


def read_text_file(path: Path):
    for encoding in ("utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"):
        try:
            return path.read_text(encoding=encoding)
        except (OSError, UnicodeDecodeError):
            continue
    return None


def count_words_general(text: str) -> int:
    return len(re.findall(r"\S+", text)) if text else 0


def get_bin_label(count: int, bin_size: int = BIN_SIZE) -> str:
    start = (count // bin_size) * bin_size
    return f"{start}-{start + bin_size}"


file_rows = []
for group, folder_path in FOLDERS.items():
    if not folder_path.exists():
        print(f"WARNING: Folder not found: {folder_path}")
        continue

    for path in sorted(folder_path.iterdir()):
        if not path.is_file():
            continue
        if group in ARABIC_FOLDER_KEYS and path.suffix.lower() != ".txt":
            continue

        text = read_text_file(path)
        if text is None:
            file_rows.append(
                {
                    "group": group,
                    "folder_path": str(folder_path),
                    "filename": path.name,
                    "extension": path.suffix.lower(),
                    "word_count": None,
                    "word_range": "UNREADABLE",
                    "status": "UNREADABLE",
                }
            )
            continue

        word_count = count_words_general(text)
        file_rows.append(
            {
                "group": group,
                "folder_path": str(folder_path),
                "filename": path.name,
                "extension": path.suffix.lower(),
                "word_count": word_count,
                "word_range": get_bin_label(word_count),
                "status": "OK",
            }
        )

file_columns = [
    "group",
    "folder_path",
    "filename",
    "extension",
    "word_count",
    "word_range",
    "status",
]
files_df = pd.DataFrame(file_rows, columns=file_columns)
ok_df = files_df.loc[files_df["status"] == "OK"].copy()

folder_summary = ok_df.groupby("group", as_index=False).agg(
    total_words=("word_count", "sum"),
    file_count=("filename", "count"),
    average_words_per_file=("word_count", "mean"),
    min_words=("word_count", "min"),
    max_words=("word_count", "max"),
)
folder_summary["average_words_per_file"] = folder_summary[
    "average_words_per_file"
].round(2)

range_counts = (
    ok_df.groupby(["group", "word_range"], as_index=False)
    .size()
    .rename(columns={"size": "file_count"})
)

if range_counts.empty:
    range_pivot = pd.DataFrame({"word_range": pd.Series(dtype=str)})
else:
    range_pivot = (
        range_counts.pivot(index="word_range", columns="group", values="file_count")
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    range_pivot["range_start"] = (
        range_pivot["word_range"].str.split("-").str[0].astype(int)
    )
    range_pivot = range_pivot.sort_values("range_start").drop(columns="range_start")

print("=" * 90)
print("TOTAL WORDS BY SUBFOLDER")
print("=" * 90)
display(folder_summary)

print("\n" + "=" * 90)
print("FILES BY WORD-COUNT RANGE")
print("=" * 90)
display(range_pivot)

print("\n" + "=" * 90)
print("PLAIN TEXT SUMMARY")
print("=" * 90)
for _, row in folder_summary.iterrows():
    print(f"{row['group']}:")
    print(f"  file_count = {int(row['file_count'])}")
    print(f"  total_words = {int(row['total_words'])}")
    print(f"  average_words_per_file = {row['average_words_per_file']}")
    print(f"  min_words = {int(row['min_words'])}")
    print(f"  max_words = {int(row['max_words'])}")

print("\n" + "=" * 90)
print("PLAIN TEXT RANGE COUNTS")
print("=" * 90)
for _, row in range_pivot.iterrows():
    print(f"{row['word_range']}:")
    for group in (column for column in range_pivot.columns if column != "word_range"):
        print(f"  {group}: {int(row[group])}")

files_csv = REPORT_DIR / "file_level_word_counts.csv"
summary_csv = REPORT_DIR / "folder_word_summary.csv"
ranges_csv = REPORT_DIR / "word_range_distribution.csv"
excel_path = REPORT_DIR / "word_count_reports.xlsx"

files_df.to_csv(files_csv, index=False, encoding="utf-8-sig")
folder_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
range_pivot.to_csv(ranges_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    files_df.to_excel(writer, sheet_name="file_level_counts", index=False)
    folder_summary.to_excel(writer, sheet_name="folder_summary", index=False)
    range_pivot.to_excel(writer, sheet_name="range_distribution", index=False)

print("\n" + "=" * 90)
print("SAVED REPORTS")
print("=" * 90)
print(files_csv)
print(summary_csv)
print(ranges_csv)
print(excel_path)

In [ ]:
import base64
import json
import os
import re
import tempfile
import time
from pathlib import Path

import fitz
import pandas as pd
from openai import OpenAI

# Configure PROJECT_DATA_PATH and OPENAI_API_KEY outside the notebook.
BASE = Path(os.environ["PROJECT_DATA_PATH"])
PDF_DIR = BASE / "AI-Free Assignments"
OUTPUT_DIR = BASE / "Trimmed AI-Free Assignments Direct PDF OCR Strong"
RAW_OCR_JSON_DIR = OUTPUT_DIR / "_raw_ocr_json"
KEEP_JSON_DIR = OUTPUT_DIR / "_keep_ids_json"
REVIEW_DIR = OUTPUT_DIR / "_review"
REPORT_XLSX = BASE / "direct_pdf_ocr_strong_trim_report.xlsx"

for directory in (OUTPUT_DIR, RAW_OCR_JSON_DIR, KEEP_JSON_DIR, REVIEW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL = "gpt-4.1-mini"
OPENAI_API_KEY = globals().get("OPENAI_API_KEY", "") or os.environ.get(
    "OPENAI_API_KEY", ""
)

RESUME_MODE = True
MAX_RETRIES = 5
SLEEP_BETWEEN_CALLS = 1.0
OCR_PAGES_PER_BATCH = 6
MAX_FINAL_WORDS = 3000
MIN_FINAL_WORDS = 700


def require_key():
    if not OPENAI_API_KEY.strip():
        raise ValueError("Set OPENAI_API_KEY before running this pipeline.")


def list_pdfs(folder: Path):
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return sorted(
        (
            path
            for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() == ".pdf"
        ),
        key=lambda path: path.name.lower(),
    )


def count_words(text: str):
    return len(re.findall(r"\b[\w\u0600-\u06FF]+\b", text, flags=re.UNICODE))


def clean_text(text: str):
    text = (text or "").replace("\ufeff", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def read_text(path: Path):
    for encoding in ("utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"):
        try:
            return path.read_text(encoding=encoding, errors="ignore")
        except Exception:
            pass
    return ""


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def pdf_to_base64_data_url(pdf_path: Path):
    encoded = base64.b64encode(pdf_path.read_bytes()).decode("utf-8")
    return f"data:application/pdf;base64,{encoded}"


def create_pdf_batch(src_pdf: Path, start_page: int, end_page: int, out_pdf: Path):
    source = fitz.open(src_pdf)
    batch = fitz.open()
    try:
        batch.insert_pdf(source, from_page=start_page, to_page=end_page)
        batch.save(out_pdf)
    finally:
        batch.close()
        source.close()


def parse_structured_output(client, *, input_payload, schema_name, schema):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=input_payload,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "schema": schema,
                        "strict": True,
                    }
                },
            )

            if getattr(response, "status", None) == "incomplete":
                details = getattr(response, "incomplete_details", None)
                reason = getattr(details, "reason", "unknown")
                raise RuntimeError(f"Response incomplete: {reason}")

            output = (getattr(response, "output_text", "") or "").strip()
            if not output:
                raise RuntimeError("Empty model output.")
            return json.loads(output)
        except Exception as error:
            last_error = error
            print(f"    retry {attempt}/{MAX_RETRIES}: {error}")
            time.sleep(min(20, 2 * attempt))

    raise last_error


OCR_SCHEMA = {
    "type": "object",
    "properties": {
        "pages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "page_num": {"type": "integer"},
                    "paragraphs": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["page_num", "paragraphs"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["pages"],
    "additionalProperties": False,
}


def ocr_pdf_batch_to_pages(
    client, batch_pdf_path: Path, source_name: str, start_page_num: int
):
    # Small PDF batches make direct OCR requests more stable.
    instruction = f"""
Read this Arabic academic PDF and return exact OCR paragraphs by original page number.

Do not summarize, paraphrase, rewrite, normalize, or add text. Merge wrapped lines only when they clearly form one paragraph, and split text only at clear paragraph boundaries. Remove obvious page numbers and repeated headers or footers. Preserve readable Arabic article text and correct reading order for multi-column pages.

This batch begins at original page {start_page_num}. Source file: {source_name}
""".strip()

    data = parse_structured_output(
        client,
        input_payload=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": batch_pdf_path.name,
                        "file_data": pdf_to_base64_data_url(batch_pdf_path),
                    },
                    {"type": "input_text", "text": instruction},
                ],
            }
        ],
        schema_name="ocr_pages",
        schema=OCR_SCHEMA,
    )

    pages = []
    for page in data.get("pages", []):
        paragraphs = [clean_text(text) for text in page["paragraphs"]]
        paragraphs = [text for text in paragraphs if text]
        if paragraphs:
            pages.append({"page_num": int(page["page_num"]), "paragraphs": paragraphs})
    return pages


def ocr_full_pdf_in_batches(client, pdf_path: Path):
    document = fitz.open(pdf_path)
    try:
        total_pages = len(document)
    finally:
        document.close()

    all_pages = []
    with tempfile.TemporaryDirectory() as temporary_directory:
        temporary_directory = Path(temporary_directory)
        for start in range(0, total_pages, OCR_PAGES_PER_BATCH):
            end = min(total_pages - 1, start + OCR_PAGES_PER_BATCH - 1)
            batch_pdf = temporary_directory / f"batch_{start + 1}_{end + 1}.pdf"

            print(f"  OCR pages {start + 1}-{end + 1}")
            create_pdf_batch(pdf_path, start, end, batch_pdf)
            all_pages.extend(
                ocr_pdf_batch_to_pages(client, batch_pdf, pdf_path.name, start + 1)
            )
            time.sleep(SLEEP_BETWEEN_CALLS)

    all_pages.sort(key=lambda page: page["page_num"])
    paragraph_id = 1
    pages_with_ids = []
    for page in all_pages:
        paragraphs = []
        for text in page["paragraphs"]:
            paragraphs.append({"id": paragraph_id, "text": text})
            paragraph_id += 1
        pages_with_ids.append({"page_num": page["page_num"], "paragraphs": paragraphs})

    if not pages_with_ids:
        raise RuntimeError("No OCR pages returned.")
    return {"pages": pages_with_ids}


KEEP_SCHEMA = {
    "type": "object",
    "properties": {"keep_ids": {"type": "array", "items": {"type": "integer"}}},
    "required": ["keep_ids"],
    "additionalProperties": False,
}

DROP_SCHEMA = {
    "type": "object",
    "properties": {"drop_ids": {"type": "array", "items": {"type": "integer"}}},
    "required": ["drop_ids"],
    "additionalProperties": False,
}


def flatten_ocr_pages(ocr_json):
    return [
        {
            "id": paragraph["id"],
            "page_num": page["page_num"],
            "word_count": count_words(paragraph["text"]),
            "text": paragraph["text"],
        }
        for page in ocr_json["pages"]
        for paragraph in page["paragraphs"]
    ]


def select_keep_ids(client, ocr_json: dict, source_name: str):
    instruction = f"""
Select exact OCR paragraph IDs from an Arabic academic PDF for AI-detection benchmarking. The final text is copied verbatim in original order and must not exceed {MAX_FINAL_WORDS} words.

Keep coherent article content such as a meaningful title, abstract, useful headings, body paragraphs, and conclusions. Exclude headers, footers, page numbers, publication metadata, affiliations, emails, DOI or date lines, references, repeated titles, and unreadable fragments. Do not rewrite, summarize, or invent text.

Source file: {source_name}
""".strip()

    data = parse_structured_output(
        client,
        input_payload=[
            {
                "role": "user",
                "content": instruction
                + "\n\nParagraph inventory:\n"
                + json.dumps(flatten_ocr_pages(ocr_json), ensure_ascii=False, indent=2),
            }
        ],
        schema_name="keep_ids",
        schema=KEEP_SCHEMA,
    )
    return sorted(
        {
            int(value)
            for value in data.get("keep_ids", [])
            if isinstance(value, int) or str(value).isdigit()
        }
    )


def build_final_text_from_keep_ids(ocr_pages, keep_ids):
    keep_set = set(keep_ids)
    paragraphs = [
        clean_text(paragraph["text"])
        for page in ocr_pages
        for paragraph in page["paragraphs"]
        if paragraph["id"] in keep_set and clean_text(paragraph["text"])
    ]
    return "\n\n".join(paragraphs).strip() + "\n"


def reduce_to_word_limit(client, ocr_json: dict, keep_ids: list[int], source_name: str):
    if (
        count_words(build_final_text_from_keep_ids(ocr_json["pages"], keep_ids))
        <= MAX_FINAL_WORDS
    ):
        return keep_ids

    keep_set = set(keep_ids)
    kept = [
        {
            "id": paragraph["id"],
            "page_num": page["page_num"],
            "word_count": count_words(paragraph["text"]),
            "text": paragraph["text"],
        }
        for page in ocr_json["pages"]
        for paragraph in page["paragraphs"]
        if paragraph["id"] in keep_set
    ]

    instruction = f"""
Choose exact paragraph IDs to drop from already selected Arabic OCR text so the remaining verbatim text is at most {MAX_FINAL_WORDS} words. Drop as few paragraphs as possible while preserving coherence. Prefer low-value, repeated, metadata-like, or less central text. Do not rewrite or change wording.

Source file: {source_name}
""".strip()

    data = parse_structured_output(
        client,
        input_payload=[
            {
                "role": "user",
                "content": instruction
                + "\n\nCurrently kept paragraphs:\n"
                + json.dumps(kept, ensure_ascii=False, indent=2),
            }
        ],
        schema_name="drop_ids",
        schema=DROP_SCHEMA,
    )
    drop_ids = {
        int(value)
        for value in data.get("drop_ids", [])
        if isinstance(value, int) or str(value).isdigit()
    }
    return sorted(set(keep_ids) - drop_ids)


require_key()
client = OpenAI(api_key=OPENAI_API_KEY)
pdf_files = list_pdfs(PDF_DIR)
print(f"Found {len(pdf_files)} PDFs.")

report_rows = []
for index, pdf_path in enumerate(pdf_files, 1):
    print(f"\n[{index}/{len(pdf_files)}] {pdf_path.name}")

    stem = pdf_path.stem
    output_text_path = OUTPUT_DIR / f"{stem}.txt"
    raw_json_path = RAW_OCR_JSON_DIR / f"{stem}.json"
    keep_json_path = KEEP_JSON_DIR / f"{stem}.json"
    review_text_path = REVIEW_DIR / f"{stem}.txt"

    if RESUME_MODE and output_text_path.exists():
        existing = read_text(output_text_path)
        report_rows.append(
            {
                "pdf_file": pdf_path.name,
                "status": "reused_existing",
                "trimmed_txt": str(output_text_path),
                "word_count": count_words(existing),
                "kept_paragraphs": "",
                "reason": "already_exists",
            }
        )
        print("  -> reused existing output")
        continue

    try:
        if RESUME_MODE and raw_json_path.exists():
            ocr_json = json.loads(raw_json_path.read_text(encoding="utf-8"))
        else:
            print("  OCR...")
            ocr_json = ocr_full_pdf_in_batches(client, pdf_path)
            write_json(raw_json_path, ocr_json)

        print("  selecting paragraphs...")
        keep_ids = select_keep_ids(client, ocr_json, pdf_path.name)
        time.sleep(SLEEP_BETWEEN_CALLS)
        keep_ids = reduce_to_word_limit(client, ocr_json, keep_ids, pdf_path.name)
        write_json(keep_json_path, {"keep_ids": keep_ids})

        final_text = (
            clean_text(build_final_text_from_keep_ids(ocr_json["pages"], keep_ids))
            + "\n"
        )
        final_words = count_words(final_text)

        if final_words < MIN_FINAL_WORDS:
            write_text(review_text_path, final_text)
            report_rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "status": "review",
                    "trimmed_txt": str(review_text_path),
                    "word_count": final_words,
                    "kept_paragraphs": len(keep_ids),
                    "reason": "too_short_after_trimming",
                }
            )
            print(f"  -> review | too short | words={final_words}")
        else:
            write_text(output_text_path, final_text)
            report_rows.append(
                {
                    "pdf_file": pdf_path.name,
                    "status": "done",
                    "trimmed_txt": str(output_text_path),
                    "word_count": final_words,
                    "kept_paragraphs": len(keep_ids),
                    "reason": "ok",
                }
            )
            print(f"  -> done | words={final_words} | kept_paragraphs={len(keep_ids)}")
    except Exception as error:
        report_rows.append(
            {
                "pdf_file": pdf_path.name,
                "status": "failed",
                "trimmed_txt": "",
                "word_count": "",
                "kept_paragraphs": "",
                "reason": str(error),
            }
        )
        print(f"  -> FAILED: {error}")

report_df = pd.DataFrame(report_rows)
summary_df = pd.DataFrame(
    [
        {
            "pdf_count_found": len(pdf_files),
            "done_count": (
                int((report_df["status"] == "done").sum()) if not report_df.empty else 0
            ),
            "review_count": (
                int((report_df["status"] == "review").sum())
                if not report_df.empty
                else 0
            ),
            "failed_count": (
                int((report_df["status"] == "failed").sum())
                if not report_df.empty
                else 0
            ),
            "output_dir": str(OUTPUT_DIR),
            "review_dir": str(REVIEW_DIR),
        }
    ]
)

with pd.ExcelWriter(REPORT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, index=False, sheet_name="summary")
    report_df.to_excel(writer, index=False, sheet_name="file_report")

print("\nDONE")
print("Trimmed TXT output folder:", OUTPUT_DIR)
print("Raw OCR JSON folder:", RAW_OCR_JSON_DIR)
print("Keep-IDs JSON folder:", KEEP_JSON_DIR)
print("Review folder:", REVIEW_DIR)
print("Report:", REPORT_XLSX)

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("${REDACTED_SECRET}")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests

API_URL = os.environ.get("WRITEHUMAN_API_URL", "https://api.writehuman.ai/v1/humanize")
API_KEY = os.environ["WRITEHUMAN_API_KEY"]
INPUT_DIR = Path(os.environ["PIPELINE_INPUT_DIR"]).expanduser()
OUTPUT_DIR = Path(os.environ["PIPELINE_OUTPUT_DIR"]).expanduser()

TIMEOUT_SECONDS = 120
MAX_RETRIES = 5
SLEEP_BETWEEN_REQUESTS = 1.0
OVERWRITE_EXISTING = False
MAX_WORDS_PER_REQUEST = None

input_dir = INPUT_DIR.resolve()
output_dir = OUTPUT_DIR.resolve()
output_dir.mkdir(parents=True, exist_ok=True)

if not input_dir.exists():
    raise FileNotFoundError(f"Input folder not found:\n{input_dir}")

if input_dir == output_dir:
    raise ValueError(
        "PIPELINE_INPUT_DIR and PIPELINE_OUTPUT_DIR must be different directories."
    )


def is_output_file(path):
    try:
        path.relative_to(output_dir)
        return True
    except ValueError:
        return False


txt_files = sorted(
    path
    for path in input_dir.rglob("*.txt")
    if path.is_file() and not is_output_file(path)
)

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in:\n{input_dir}")

session = requests.Session()
session.headers.update(
    {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
)


def count_words(text):
    return len(re.findall(r"\S+", text, flags=re.UNICODE))


def split_text_by_words(text, max_words):
    if max_words is None or max_words <= 0:
        return [text]

    # Retain whitespace while splitting requests by non-whitespace tokens.
    tokens = re.findall(r"\S+|\s+", text, flags=re.UNICODE)
    chunks = []
    current = []
    current_word_count = 0

    for token in tokens:
        if token.isspace():
            current.append(token)
            continue

        if current_word_count + 1 > max_words and current:
            chunks.append("".join(current).strip())
            current = [token]
            current_word_count = 1
        else:
            current.append(token)
            current_word_count += 1

    if current:
        chunks.append("".join(current).strip())

    return [chunk for chunk in chunks if chunk.strip()]


def call_api(text):
    payload = {"text": text}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.post(API_URL, json=payload, timeout=TIMEOUT_SECONDS)

            if response.status_code >= 400:
                try:
                    error_detail = response.json()
                except ValueError:
                    error_detail = response.text
                raise RuntimeError(f"HTTP {response.status_code}: {error_detail}")

            data = response.json()
            if not isinstance(data, dict):
                raise ValueError(f"Unexpected response type: {type(data)}")

            results = data.get("results")
            if not isinstance(results, list) or not results:
                raise ValueError(f"Response missing usable 'results': {data}")

            modified_text = results[0]
            if not isinstance(modified_text, str):
                raise ValueError(f"First result is not text: {data}")

            return modified_text, data

        except Exception as error:
            last_error = error
            if attempt == MAX_RETRIES:
                raise

            wait_seconds = min(2 ** (attempt - 1), 10)
            print(f"    Retry {attempt}/{MAX_RETRIES} after error: {error}")
            time.sleep(wait_seconds)

    raise last_error


def process_file(src_path, dst_path):
    with src_path.open("r", encoding="utf-8-sig", errors="ignore") as file:
        original_text = file.read().strip()

    if not original_text:
        raise ValueError("File is empty after reading.")

    chunks = split_text_by_words(original_text, MAX_WORDS_PER_REQUEST)
    modified_chunks = []
    last_response = {}

    for index, chunk in enumerate(chunks, start=1):
        print(f"      Chunk {index}/{len(chunks)} | words={count_words(chunk)}")
        modified_text, last_response = call_api(chunk)
        modified_chunks.append(modified_text.strip())
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    final_text = "\n\n".join(modified_chunks).strip()

    with dst_path.open("w", encoding="utf-8") as file:
        file.write(final_text)

    return {
        "input_words": count_words(original_text),
        "output_words": count_words(final_text),
        "chunks_sent": len(chunks),
        "api_last_response": last_response,
    }


records = []
success_count = 0
fail_count = 0

print(f"Found {len(txt_files)} txt files.")
print(f"Input:  {input_dir}")
print(f"Output: {output_dir}\n")

for index, src_path in enumerate(txt_files, start=1):
    relative_path = src_path.relative_to(input_dir)
    dst_path = output_dir / relative_path
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if dst_path.exists() and not OVERWRITE_EXISTING:
        print(f"[{index}/{len(txt_files)}] SKIP (already exists): {relative_path}")
        records.append(
            {
                "file": str(relative_path),
                "status": "skipped_existing",
                "input_words": None,
                "output_words": None,
                "chunks_sent": None,
                "error": "",
            }
        )
        continue

    print(f"[{index}/{len(txt_files)}] Processing: {relative_path}")

    try:
        metadata = process_file(src_path, dst_path)
        success_count += 1
        records.append(
            {
                "file": str(relative_path),
                "status": "success",
                "input_words": metadata["input_words"],
                "output_words": metadata["output_words"],
                "chunks_sent": metadata["chunks_sent"],
                "error": "",
            }
        )
        print(
            f"    Done | input_words={metadata['input_words']} "
            f"| output_words={metadata['output_words']}"
        )
    except Exception as error:
        fail_count += 1
        records.append(
            {
                "file": str(relative_path),
                "status": "failed",
                "input_words": None,
                "output_words": None,
                "chunks_sent": None,
                "error": str(error),
            }
        )
        print(f"    FAILED: {error}")

    print()

log_path = output_dir / "_processing_log.csv"
pd.DataFrame(records).to_csv(log_path, index=False, encoding="utf-8-sig")

print("Finished.")
print(f"Successful: {success_count}")
print(f"Failed:     {fail_count}")
print(f"Log saved:  {log_path}")
print(f"Outputs in: {output_dir}")

In [ ]:
# OCR Arabic PDFs with Google Document AI and OpenAI
!pip -q install google-cloud-documentai openai pymupdf pandas

import os
import re
import shutil
import time
from pathlib import Path

import fitz
import pandas as pd
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
from openai import OpenAI

# Set these environment variables before running:
# INPUT_PDF_DIR, OUTPUT_BASE_DIR, GOOGLE_SERVICE_ACCOUNT_JSON,
# GCP_PROJECT_ID, DOC_AI_PROCESSOR_ID, and OPENAI_API_KEY.
MOUNT_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = "/content/drive"

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(DRIVE_MOUNT_POINT)


def require_env(name: str) -> str:
    value = os.environ.get(name)
    if not value:
        raise EnvironmentError(f"Missing required environment variable: {name}")
    return value


INPUT_PDF_DIR = Path(require_env("INPUT_PDF_DIR"))
OUTPUT_BASE_DIR = Path(require_env("OUTPUT_BASE_DIR"))
GOOGLE_SERVICE_ACCOUNT_JSON = require_env("GOOGLE_SERVICE_ACCOUNT_JSON")
GCP_PROJECT_ID = require_env("GCP_PROJECT_ID")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us")
DOC_AI_PROCESSOR_ID = require_env("DOC_AI_PROCESSOR_ID")
OPENAI_API_KEY = require_env("OPENAI_API_KEY")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.4-mini")

OVERWRITE_EXISTING = False
SLEEP_BETWEEN_OPENAI_CALLS = 1.0
SLEEP_BETWEEN_DOC_AI_CALLS = 0.4
DOC_AI_MAX_PAGES_PER_CHUNK = 15
FINAL_WORD_CAP = 6000
OPENAI_CLEAN_CHUNK_WORDS = 3500

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GOOGLE_SERVICE_ACCOUNT_JSON

input_dir = INPUT_PDF_DIR
output_base = OUTPUT_BASE_DIR
raw_ocr_dir = output_base / "_raw_ocr"
clean_full_dir = output_base / "_cleaned_full"
final_dir = output_base / "final_txt"
tmp_pdf_chunk_dir = output_base / "_tmp_pdf_chunks"
log_csv_path = output_base / "_processing_log.csv"

for directory in (output_base, raw_ocr_dir, clean_full_dir, final_dir, tmp_pdf_chunk_dir):
    directory.mkdir(parents=True, exist_ok=True)

if not input_dir.exists():
    raise FileNotFoundError(f"Input folder does not exist:\n{input_dir}")

pdf_files = sorted(path for path in input_dir.rglob("*.pdf") if path.is_file())
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in:\n{input_dir}")

docai_client = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(
        api_endpoint=f"{GCP_LOCATION}-documentai.googleapis.com"
    )
)
processor_name = docai_client.processor_path(
    GCP_PROJECT_ID,
    GCP_LOCATION,
    DOC_AI_PROCESSOR_ID,
)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

ARABIC_LETTERS_RE = re.compile(r"[\u0600-\u06FF]")


def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text or "", flags=re.UNICODE))


def normalize_spaces(text: str) -> str:
    text = text.replace("\u00A0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def has_arabic(text: str) -> bool:
    return bool(ARABIC_LETTERS_RE.search(text or ""))


def text_anchor_to_str(document_text, text_anchor):
    if not text_anchor or not text_anchor.text_segments:
        return ""

    pieces = []
    for segment in text_anchor.text_segments:
        start = int(segment.start_index) if segment.start_index is not None else 0
        end = int(segment.end_index) if segment.end_index is not None else 0
        pieces.append(document_text[start:end])
    return "".join(pieces)


def extract_structured_text_from_docai(document):
    # Paragraphs retain document structure better than the full OCR stream.
    pages_out = []
    full_text = document.text or ""

    for page in getattr(document, "pages", []):
        blocks = []
        for paragraph in getattr(page, "paragraphs", []):
            text = normalize_spaces(
                text_anchor_to_str(full_text, paragraph.layout.text_anchor)
            )
            if text:
                blocks.append(text)

        if not blocks:
            for line in getattr(page, "lines", []):
                text = normalize_spaces(
                    text_anchor_to_str(full_text, line.layout.text_anchor)
                )
                if text:
                    blocks.append(text)

        if blocks:
            pages_out.append("\n\n".join(blocks))

    return "\n\n".join(pages_out).strip() or normalize_spaces(full_text)


def split_pdf_into_chunks(pdf_path: Path, out_dir: Path, max_pages: int):
    document = fitz.open(pdf_path)
    total_pages = document.page_count
    chunk_paths = []

    try:
        for start in range(0, total_pages, max_pages):
            end = min(start + max_pages, total_pages)
            chunk = fitz.open()
            try:
                chunk.insert_pdf(document, from_page=start, to_page=end - 1)
                chunk_path = out_dir / (
                    f"{pdf_path.stem}__pages_{start + 1:03d}_{end:03d}.pdf"
                )
                chunk.save(chunk_path)
                chunk_paths.append(chunk_path)
            finally:
                chunk.close()
    finally:
        document.close()

    return chunk_paths, total_pages


def ocr_pdf_chunk_with_docai(pdf_chunk_path: Path) -> str:
    pdf_bytes = pdf_chunk_path.read_bytes()
    request = documentai.ProcessRequest(
        name=processor_name,
        raw_document=documentai.RawDocument(
            content=pdf_bytes,
            mime_type="application/pdf",
        ),
    )
    result = docai_client.process_document(request=request)
    return normalize_spaces(extract_structured_text_from_docai(result.document))


def basic_post_ocr_cleanup(text: str) -> str:
    lines = [line.strip() for line in text.splitlines()]
    cleaned = []
    previous = None

    for line in lines:
        if not line:
            cleaned.append("")
            previous = line
            continue

        if re.fullmatch(r"[\d\W_]+", line):
            continue

        if previous == line and len(line) <= 40:
            continue

        cleaned.append(line)
        previous = line

    return re.sub(r"\n{3,}", "\n\n", "\n".join(cleaned)).strip()


def split_text_paragraphwise(text: str, max_words: int):
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", normalize_spaces(text))
        if paragraph.strip()
    ]
    chunks = []
    current = []
    current_words = 0

    for paragraph in paragraphs:
        words = count_words(paragraph)
        if current and current_words + words > max_words:
            chunks.append("\n\n".join(current).strip())
            current = [paragraph]
            current_words = words
        else:
            current.append(paragraph)
            current_words += words

    if current:
        chunks.append("\n\n".join(current).strip())

    final_chunks = []
    for chunk in chunks:
        if count_words(chunk) <= max_words:
            final_chunks.append(chunk)
            continue

        tokens = re.findall(r"\S+|\s+", chunk, flags=re.UNICODE)
        current = []
        word_count = 0
        for token in tokens:
            if token.isspace():
                current.append(token)
                continue
            if word_count + 1 > max_words and current:
                final_chunks.append("".join(current).strip())
                current = [token]
                word_count = 1
            else:
                current.append(token)
                word_count += 1
        if current:
            final_chunks.append("".join(current).strip())

    return [chunk for chunk in final_chunks if chunk]


def openai_clean_ocr_chunk(text_chunk: str) -> str:
    prompt = f"""
Clean this OCR-extracted Arabic academic text while preserving its meaning and order.

- Output Arabic text only.
- Correct OCR errors only when the intended Arabic is clear.
- Join lines belonging to the same paragraph.
- Remove repeated headers, footers, page numbers, isolated garbage symbols, duplicated OCR fragments, and scanning artifacts.
- Preserve headings, bullets, and complete paragraphs.
- Do not summarize, translate, stylistically rewrite, or intentionally shorten the text.
- Do not add labels, explanations, or Markdown.

TEXT:
{text_chunk}
""".strip()

    response = openai_client.responses.create(model=OPENAI_MODEL, input=prompt)
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return normalize_spaces((response.output_text or "").strip())


def openai_trim_to_cap(cleaned_text: str, word_cap: int) -> str:
    prompt = f"""
Prepare this cleaned Arabic academic text for AI-detection input.

- Retain coherent Arabic prose, meaningful headings, and meaningful bullet points.
- Remove title pages, author metadata, contents pages, repeated headers and footers, page numbers, references, bibliographies, appendices, URLs, isolated equation or symbol blocks, and isolated figure or table captions.
- Preserve complete paragraphs and the original order of retained sections.
- Do not summarize or paraphrase retained text except for minimal readability fixes.
- If substantive prose is already below the cap, retain it after removing non-content material.
- Output no more than {word_cap} words.
- Output Arabic text only, without explanations or Markdown.

TEXT:
{cleaned_text}
""".strip()

    response = openai_client.responses.create(model=OPENAI_MODEL, input=prompt)
    time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
    return normalize_spaces((response.output_text or "").strip())


def hard_cap_to_full_paragraphs(text: str, word_cap: int) -> str:
    # Enforce the cap locally if the model exceeds it.
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", normalize_spaces(text))
        if paragraph.strip()
    ]
    kept = []
    total_words = 0

    for paragraph in paragraphs:
        words = count_words(paragraph)
        if kept and total_words + words > word_cap:
            break
        if not kept and words > word_cap:
            tokens = re.findall(r"\S+|\s+", paragraph, flags=re.UNICODE)
            result = []
            word_count = 0
            for token in tokens:
                if token.isspace():
                    result.append(token)
                elif word_count < word_cap:
                    result.append(token)
                    word_count += 1
                else:
                    break
            return normalize_spaces("".join(result))
        kept.append(paragraph)
        total_words += words

    return normalize_spaces("\n\n".join(kept))


def save_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


records = []

print(f"Found {len(pdf_files)} PDF files.")
print(f"Input:  {input_dir}")
print(f"Output: {output_base}\n")

for index, pdf_path in enumerate(pdf_files, start=1):
    relative_path = pdf_path.relative_to(input_dir)
    output_relative_path = relative_path.with_suffix(".txt")

    # Preserve relative paths to avoid collisions between PDFs with the same stem.
    raw_out = raw_ocr_dir / output_relative_path
    clean_out = clean_full_dir / output_relative_path
    final_out = final_dir / output_relative_path
    file_tmp_dir = tmp_pdf_chunk_dir / relative_path.with_suffix("")

    row = {
        "file": str(relative_path),
        "status": "",
        "pdf_pages": None,
        "raw_ocr_words": None,
        "clean_words": None,
        "final_words": None,
        "error": "",
    }

    if final_out.exists() and not OVERWRITE_EXISTING:
        print(f"[{index}/{len(pdf_files)}] SKIP: {relative_path}")
        row["status"] = "skipped_existing"
        records.append(row)
        continue

    print(f"[{index}/{len(pdf_files)}] Processing: {relative_path}")

    try:
        if file_tmp_dir.exists():
            shutil.rmtree(file_tmp_dir)
        file_tmp_dir.mkdir(parents=True, exist_ok=True)

        chunk_paths, total_pages = split_pdf_into_chunks(
            pdf_path,
            file_tmp_dir,
            DOC_AI_MAX_PAGES_PER_CHUNK,
        )
        row["pdf_pages"] = total_pages
        print(f"    PDF pages: {total_pages} | OCR chunks: {len(chunk_paths)}")

        ocr_chunks = []
        for chunk_index, chunk_path in enumerate(chunk_paths, start=1):
            print(f"      OCR chunk {chunk_index}/{len(chunk_paths)}: {chunk_path.name}")
            chunk_text = ocr_pdf_chunk_with_docai(chunk_path)
            if chunk_text:
                ocr_chunks.append(chunk_text)
            time.sleep(SLEEP_BETWEEN_DOC_AI_CALLS)

        raw_ocr_text = basic_post_ocr_cleanup("\n\n".join(ocr_chunks))
        if not raw_ocr_text:
            raise RuntimeError("OCR returned empty text.")
        if not has_arabic(raw_ocr_text):
            raise RuntimeError("OCR output does not appear to contain Arabic text.")

        save_text(raw_out, raw_ocr_text)
        row["raw_ocr_words"] = count_words(raw_ocr_text)
        print(f"    Raw OCR words: {row['raw_ocr_words']}")

        clean_input_chunks = split_text_paragraphwise(
            raw_ocr_text,
            OPENAI_CLEAN_CHUNK_WORDS,
        )
        cleaned_chunks = []
        print(f"    OpenAI clean chunks: {len(clean_input_chunks)}")

        for chunk_index, chunk in enumerate(clean_input_chunks, start=1):
            print(
                f"      Clean chunk {chunk_index}/{len(clean_input_chunks)} "
                f"| words={count_words(chunk)}"
            )
            cleaned = openai_clean_ocr_chunk(chunk)
            if cleaned:
                cleaned_chunks.append(cleaned)

        cleaned_full_text = normalize_spaces("\n\n".join(cleaned_chunks))
        if not cleaned_full_text:
            raise RuntimeError("OpenAI cleanup returned empty text.")

        save_text(clean_out, cleaned_full_text)
        row["clean_words"] = count_words(cleaned_full_text)
        print(f"    Cleaned full words: {row['clean_words']}")

        final_text = hard_cap_to_full_paragraphs(
            openai_trim_to_cap(cleaned_full_text, FINAL_WORD_CAP),
            FINAL_WORD_CAP,
        )
        if not final_text:
            raise RuntimeError("Final trimmed text is empty.")
        if not has_arabic(final_text):
            raise RuntimeError("Final trimmed text does not appear to contain Arabic text.")

        save_text(final_out, final_text)
        row["final_words"] = count_words(final_text)
        row["status"] = "success"
        print(f"    Final words: {row['final_words']}")
        print(f"    Saved: {final_out}")

    except Exception as error:
        row["status"] = "failed"
        row["error"] = str(error)
        print(f"    FAILED: {error}")

    finally:
        if file_tmp_dir.exists():
            shutil.rmtree(file_tmp_dir)

    records.append(row)
    print()

df = pd.DataFrame(records)
df.to_csv(log_csv_path, index=False, encoding="utf-8-sig")

print("=" * 40)
print("Done.")
print(f"Processed files: {len(records)}")
print(f"Success: {(df['status'] == 'success').sum()}")
print(f"Failed:  {(df['status'] == 'failed').sum()}")
print(f"Skipped: {(df['status'] == 'skipped_existing').sum()}")
print(f"Log:     {log_csv_path}")
print(f"Final TXT folder: {final_dir}")
print("=" * 40)

In [ ]:
!pip -q install openai pandas

import os
import time
from pathlib import Path

import pandas as pd
from google.colab import drive
from openai import OpenAI

drive.mount("/content/drive")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
MODEL_NAME = "gpt-5.4-mini"

INPUT_DIR = Path(os.environ.get("REWRITE_INPUT_DIR", "input"))
OUTPUT_DIR = Path(os.environ.get("REWRITE_OUTPUT_DIR", str(INPUT_DIR / "rewritten")))

MODIFICATION_INSTRUCTION = """
Without changing the length of this code, humanize it so that it looks human-written.
Keep the word count close to the original. The new files must not contain fewer than 800 words.
Do not change functionality; only revise the presentation of the code.
""".strip()

OVERWRITE_EXISTING = False
SLEEP_BETWEEN_FILES = 1.0
MAX_RETRIES = 4

if not OPENAI_API_KEY:
    raise EnvironmentError("Set the OPENAI_API_KEY environment variable before running this notebook.")

input_dir = INPUT_DIR.resolve()
output_dir = OUTPUT_DIR.resolve()

if input_dir == output_dir:
    raise ValueError("INPUT_DIR and OUTPUT_DIR must be different directories.")

if not input_dir.exists():
    raise FileNotFoundError(f"Input folder not found:\n{input_dir}")

output_dir.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=OPENAI_API_KEY)


def is_within(path: Path, directory: Path) -> bool:
    try:
        path.relative_to(directory)
        return True
    except ValueError:
        return False


def detect_code_language(text: str) -> str:
    text_lower = text.lower()
    if "import java" in text_lower or "public class" in text_lower:
        return "Java"
    if "#include" in text_lower or "int main(" in text_lower:
        return "C/C++"
    if (
        "function " in text_lower
        or "console.log(" in text_lower
        or "const " in text_lower
        or "let " in text_lower
    ):
        return "JavaScript"
    if "def " in text_lower or "import " in text_lower or "print(" in text_lower:
        return "Python"
    if (
        "matlab" in text_lower
        or "clc;" in text_lower
        or "clear;" in text_lower
        or "disp(" in text_lower
    ):
        return "MATLAB"
    return "Unknown"


def rewrite_code(code_text: str) -> str:
    language = detect_code_language(code_text)
    prompt = f"""
You are rewriting source code.

Language: {language}

User instruction:
{MODIFICATION_INSTRUCTION}

Rules:
- Preserve the original functionality.
- Preserve compilability or runnability as much as possible.
- Do not explain anything.
- Do not wrap the answer in markdown fences.
- Output only the rewritten code.

Original code:
{code_text}
""".strip()

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(model=MODEL_NAME, input=prompt)
            rewritten_code = (response.output_text or "").strip()
            if not rewritten_code:
                raise ValueError("Empty response returned by model.")
            return rewritten_code
        except Exception as error:
            last_error = error
            if attempt == MAX_RETRIES:
                raise

            wait_seconds = min(2 ** (attempt - 1), 10)
            print(f"    Retry {attempt}/{MAX_RETRIES} after error: {error}")
            time.sleep(wait_seconds)

    raise last_error


txt_files = sorted(
    path
    for path in input_dir.rglob("*.txt")
    if path.is_file() and not is_within(path.resolve(), output_dir)
)

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in:\n{input_dir}")

records = []
success_count = 0
fail_count = 0

print(f"Found {len(txt_files)} txt files.")
print(f"Input:  {input_dir}")
print(f"Output: {output_dir}")
print()

for index, source_path in enumerate(txt_files, start=1):
    relative_path = source_path.relative_to(input_dir)
    destination_path = output_dir / relative_path
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if destination_path.exists() and not OVERWRITE_EXISTING:
        print(f"[{index}/{len(txt_files)}] SKIP (already exists): {relative_path}")
        records.append(
            {
                "file": str(relative_path),
                "status": "skipped_existing",
                "language_guess": "",
                "input_chars": None,
                "output_chars": None,
                "error": "",
            }
        )
        continue

    print(f"[{index}/{len(txt_files)}] Processing: {relative_path}")

    try:
        with source_path.open("r", encoding="utf-8-sig", errors="ignore") as file:
            original_code = file.read().strip()

        if not original_code:
            raise ValueError("File is empty.")

        language_guess = detect_code_language(original_code)
        rewritten_code = rewrite_code(original_code)

        with destination_path.open("w", encoding="utf-8") as file:
            file.write(rewritten_code)

        success_count += 1
        records.append(
            {
                "file": str(relative_path),
                "status": "success",
                "language_guess": language_guess,
                "input_chars": len(original_code),
                "output_chars": len(rewritten_code),
                "error": "",
            }
        )

        print(f"    Done | language={language_guess} | saved to {destination_path}")
        time.sleep(SLEEP_BETWEEN_FILES)

    except Exception as error:
        fail_count += 1
        records.append(
            {
                "file": str(relative_path),
                "status": "failed",
                "language_guess": "",
                "input_chars": None,
                "output_chars": None,
                "error": str(error),
            }
        )
        print(f"    FAILED: {error}")

    print()

log_path = output_dir / "_rewrite_log.csv"
pd.DataFrame(records).to_csv(log_path, index=False, encoding="utf-8-sig")

print("Finished.")
print(f"Successful: {success_count}")
print(f"Failed:     {fail_count}")
print(f"Log saved:  {log_path}")
print(f"Outputs in: {output_dir}")

In [ ]:
import math
import os
import re
from pathlib import Path

import pandas as pd

PROJECT_DATA_PATH = os.environ.get("PROJECT_DATA_PATH")
if not PROJECT_DATA_PATH:
    raise EnvironmentError(
        "Set PROJECT_DATA_PATH to the directory containing the research data."
    )

BASE_TESTING_SAMPLE_DIR = Path(PROJECT_DATA_PATH)

FOLDERS = {
    "Arabic_AI_Free": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "Trimmed AI-Free Assignments",
    "Arabic_AI_Generated": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "AI-Generated Assignments",
    "Arabic_Humanized": BASE_TESTING_SAMPLE_DIR
    / "Arabic Assignments"
    / "Humanized AI Assignments",
    "Code_AI_Free": BASE_TESTING_SAMPLE_DIR / "Coding Assignments" / "AI-Free Code",
    "Code_AI_Generated": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "AI-Generated Code",
    "Code_Humanized": BASE_TESTING_SAMPLE_DIR
    / "Coding Assignments"
    / "Humanized AI Code",
}

BIN_SIZE = 1000
LOW_WORD_THRESHOLD = 800


def count_words(text: str) -> int:
    # Whitespace-delimited tokens provide a language-agnostic count for Arabic and code files.
    return len(re.findall(r"\S+", text or "", flags=re.UNICODE))


def bin_label(word_count: int, bin_size: int = BIN_SIZE) -> str:
    lower = (word_count // bin_size) * bin_size
    upper = lower + bin_size
    return f"{lower // 1000}k-{upper // 1000}k" if lower > 0 else "0-1k"


def sample_type(folder_label: str) -> str:
    if "AI_Free" in folder_label:
        return "AI_Free"
    if "AI_Generated" in folder_label:
        return "AI_Generated"
    return "Humanized"


def folder_records(folder_label: str, folder_path: Path) -> list[dict]:
    if not folder_path.exists():
        print(f"WARNING: folder not found -> {folder_path}")
        return []

    records = []
    for file_path in sorted(
        path for path in folder_path.rglob("*.txt") if path.is_file()
    ):
        record = {
            "group": folder_label,
            "category": "Arabic" if folder_label.startswith("Arabic_") else "Code",
            "sample_type": sample_type(folder_label),
            "file_name": file_path.name,
            "relative_path": str(file_path.relative_to(folder_path)),
        }

        try:
            record["word_count"] = count_words(
                file_path.read_text(encoding="utf-8-sig", errors="ignore")
            )
            record["range"] = bin_label(record["word_count"])
        except OSError as error:
            record["word_count"] = None
            record["range"] = "read_error"
            print(f"WARNING: failed reading {file_path}: {error}")

        records.append(record)

    return records


def make_range_order(max_words: int, bin_size: int = BIN_SIZE) -> list[str]:
    max_upper = max(bin_size, int(math.ceil(max_words / bin_size) * bin_size))
    return [
        f"{lower // 1000}k-{(lower + bin_size) // 1000}k" if lower > 0 else "0-1k"
        for lower in range(0, max_upper, bin_size)
    ]


def summarize_distribution(
    df_subset: pd.DataFrame,
    summary_name: str,
    range_order: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    valid = df_subset[df_subset["word_count"].notna()].copy()
    total_files = len(valid)
    total_words = int(valid["word_count"].sum()) if total_files else 0
    average_words = float(valid["word_count"].mean()) if total_files else 0.0
    counts = valid["range"].value_counts().to_dict()

    distribution = pd.DataFrame(
        [
            {
                "summary_name": summary_name,
                "range": word_range,
                "files_in_range": int(counts.get(word_range, 0)),
                "percent_of_files": round(
                    (
                        100.0 * counts.get(word_range, 0) / total_files
                        if total_files
                        else 0.0
                    ),
                    2,
                ),
            }
            for word_range in range_order
        ]
    )

    metadata = pd.DataFrame(
        [
            {
                "summary_name": summary_name,
                "num_files": total_files,
                "total_words": total_words,
                "average_words_per_file": round(average_words, 2),
            }
        ]
    )

    return metadata, distribution


all_records = []
for label, path in FOLDERS.items():
    all_records.extend(folder_records(label, path))

files_df = pd.DataFrame(all_records)
if files_df.empty:
    raise ValueError("No TXT files were found in the configured folders.")

valid_df = files_df[files_df["word_count"].notna()].copy()
max_words_found = int(valid_df["word_count"].max()) if not valid_df.empty else 0
range_order = make_range_order(max_words_found)

meta_frames = []
distribution_frames = []

for group in files_df["group"].dropna().unique():
    metadata, distribution = summarize_distribution(
        files_df[files_df["group"] == group], group, range_order
    )
    meta_frames.append(metadata)
    distribution_frames.append(distribution)

for category in files_df["category"].dropna().unique():
    metadata, distribution = summarize_distribution(
        files_df[files_df["category"] == category], f"{category}_TOTAL", range_order
    )
    meta_frames.append(metadata)
    distribution_frames.append(distribution)

for sample in files_df["sample_type"].dropna().unique():
    metadata, distribution = summarize_distribution(
        files_df[files_df["sample_type"] == sample], f"{sample}_TOTAL", range_order
    )
    meta_frames.append(metadata)
    distribution_frames.append(distribution)

metadata, distribution = summarize_distribution(
    files_df, "ALL_FILES_TOTAL", range_order
)
meta_frames.append(metadata)
distribution_frames.append(distribution)

summary_meta_df = pd.concat(meta_frames, ignore_index=True)
summary_dist_df = pd.concat(distribution_frames, ignore_index=True)

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

print("=" * 100)
print("SUMMARY TOTALS")
print("=" * 100)
print(summary_meta_df.to_string(index=False))

print("\n" + "=" * 100)
print("RANGE DISTRIBUTION")
print("=" * 100)
print(summary_dist_df.to_string(index=False))

print("\n" + "=" * 100)
print(f"FILES WITH LESS THAN {LOW_WORD_THRESHOLD} WORDS IN EACH SUBFOLDER")
print("=" * 100)

for group in FOLDERS:
    subset = files_df[
        (files_df["group"] == group)
        & files_df["word_count"].notna()
        & (files_df["word_count"] < LOW_WORD_THRESHOLD)
    ].copy()

    print(f"\n--- {group} ---")
    if subset.empty:
        print(f"No files under {LOW_WORD_THRESHOLD} words.")
        continue

    subset = subset.sort_values(["word_count", "file_name"])
    print(f"Number of files under {LOW_WORD_THRESHOLD} words: {len(subset)}")
    print(f"Total words across those files: {int(subset['word_count'].sum())}")
    print("\nWord count in each file:")
    print(subset[["file_name", "word_count"]].to_string(index=False))

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
DATA_ROOT = Path(os.environ["AI_DETECTION_DATA_ROOT"])
MODEL_NAME = "gpt-4.1-mini"

SOURCE_GROUPS = {
    "Arabic_AI_Free": {
        "mode": "arabic",
        "src": DATA_ROOT / "Arabic Assignments" / "Trimmed AI-Free Assignments",
    },
    "Code_AI_Free": {
        "mode": "code",
        "src": DATA_ROOT / "Coding Assignments" / "AI-Free Code",
    },
    "Code_AI_Generated": {
        "mode": "code",
        "src": DATA_ROOT / "Coding Assignments" / "AI-Generated Code",
    },
    "Code_Humanized": {
        "mode": "code",
        "src": DATA_ROOT / "Coding Assignments" / "Humanized AI Code",
    },
}

OUTPUT_BASE_DIR = DATA_ROOT / "Exact_Span_Trimmed_For_AI_Detection"
TARGETS = {
    "Arabic_AI_Free": {"min_words": 1200, "max_words": 2000, "ideal_words": 1500},
    "Code_AI_Free": {"min_words": 1000, "max_words": 1800, "ideal_words": 1300},
    "Code_AI_Generated": {"min_words": 1000, "max_words": 1800, "ideal_words": 1300},
    "Code_Humanized": {"min_words": 1000, "max_words": 1800, "ideal_words": 1300},
}

OVERWRITE_EXISTING = False
MAX_RETRIES_API = 4
MAX_SELECTION_ATTEMPTS = 3
SLEEP_BETWEEN_CALLS = 1.0
MAX_PROMPT_CHARS = 120000

client = OpenAI(api_key=OPENAI_API_KEY)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

for group_name, info in SOURCE_GROUPS.items():
    if not info["src"].exists():
        raise FileNotFoundError(
            f"Missing source folder for {group_name}:\n{info['src']}"
        )
    info["dst"] = OUTPUT_BASE_DIR / info["src"].relative_to(DATA_ROOT)
    info["dst"].mkdir(parents=True, exist_ok=True)


def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text or "", flags=re.UNICODE))


def read_raw_text(path: Path) -> str:
    # Preserve original newline sequences so selected spans are written unchanged.
    with path.open("r", encoding="utf-8-sig", errors="ignore", newline="") as handle:
        return handle.read()


def write_raw_text(path: Path, text: str) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        handle.write(text)


def prefix_word_counts(lines: list[str]) -> list[int]:
    totals = []
    total = 0
    for line in lines:
        total += count_words(line)
        totals.append(total)
    return totals


def words_between(prefix_counts: list[int], start_line: int, end_line: int) -> int:
    if start_line < 1 or end_line < start_line:
        return 0
    end_total = prefix_counts[end_line - 1]
    start_total = prefix_counts[start_line - 2] if start_line > 1 else 0
    return end_total - start_total


def build_numbered_view(lines: list[str], max_prompt_chars: int) -> tuple[str, int]:
    rows = []
    total_chars = 0

    for line_number, line in enumerate(lines, start=1):
        display = line.rstrip("\r\n")
        row = f"{line_number:05d} | cw={count_words(display):03d} | {display}\n"
        if total_chars + len(row) > max_prompt_chars:
            if not rows:
                rows.append(row[:max_prompt_chars])
            break
        rows.append(row)
        total_chars += len(row)

    return "".join(rows), len(rows)


def parse_json_response(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
    raise ValueError("Could not parse JSON from model output.")


def call_model_json(prompt: str) -> dict:
    last_error = None
    for attempt in range(1, MAX_RETRIES_API + 1):
        try:
            response = client.responses.create(model=MODEL_NAME, input=prompt)
            output = (response.output_text or "").strip()
            if not output:
                raise ValueError("Empty model output.")
            time.sleep(SLEEP_BETWEEN_CALLS)
            return parse_json_response(output)
        except Exception as error:
            last_error = error
            if attempt == MAX_RETRIES_API:
                raise
            wait_seconds = min(2 ** (attempt - 1), 12)
            print(f"      API retry {attempt}/{MAX_RETRIES_API}: {error}")
            time.sleep(wait_seconds)
    raise last_error


def extract_exact_span(
    raw_text: str, start_line: int, end_line: int
) -> tuple[str, int, int]:
    lines = raw_text.splitlines(keepends=True)
    if not lines:
        return "", 0, 0

    start_line = max(1, min(start_line, len(lines)))
    end_line = max(start_line, min(end_line, len(lines)))
    return "".join(lines[start_line - 1 : end_line]), start_line, end_line


def find_best_fallback_window(
    lines: list[str], min_words: int, max_words: int, ideal_words: int
) -> tuple[int, int]:
    if not lines:
        return 1, 1

    prefix_counts = prefix_word_counts(lines)
    best = None
    end_index = 0

    for start_index in range(len(lines)):
        end_index = max(end_index, start_index)
        while (
            end_index < len(lines)
            and words_between(prefix_counts, start_index + 1, end_index + 1) < min_words
        ):
            end_index += 1

        for candidate_end in (end_index - 1, end_index, end_index + 1):
            if candidate_end < start_index or candidate_end >= len(lines):
                continue

            word_count = words_between(
                prefix_counts, start_index + 1, candidate_end + 1
            )
            score = (
                0 if min_words <= word_count <= max_words else 1,
                abs(word_count - ideal_words),
                candidate_end - start_index,
            )
            if best is None or score < best[0]:
                best = (score, start_index + 1, candidate_end + 1)

    return (1, len(lines)) if best is None else best[1:]


def build_prompt(
    mode: str,
    group_name: str,
    numbered_view: str,
    visible_line_count: int,
    total_line_count: int,
    source_words: int,
    min_words: int,
    max_words: int,
    ideal_words: int,
    attempt_note: str = "",
) -> str:
    if mode == "arabic":
        selection_guidance = "Select coherent, substantive Arabic prose. Avoid title pages, contents, references, appendices, metadata, repeated headers or footers, and other non-prose material. Prefer complete paragraphs and natural boundaries."
    else:
        selection_guidance = "Select meaningful code regions such as complete functions, methods, classes, or coherent logical sections. Avoid prompts, logs, dependency dumps, boilerplate, large outputs, and irrelevant prose. Prefer boundaries that do not split code blocks."

    return f"""Select one contiguous excerpt for authorship analysis.

Rules:
- Return one start line and one end line.
- The selected lines will be retained without editing.
- Keep the original order.
- Target {min_words} to {max_words} words, ideally about {ideal_words} words.
- {selection_guidance}
- Only lines 1 through {visible_line_count} are available. Select only from those lines.

Return JSON only:
{{
  "start_line": <integer>,
  "end_line": <integer>,
  "reason": "<brief reason>"
}}

{attempt_note}

Context:
- Group: {group_name}
- Total source words: {source_words}
- Total source lines: {total_line_count}

Line-numbered file view:
{numbered_view}"""


def choose_span_with_model(
    raw_text: str,
    mode: str,
    group_name: str,
    min_words: int,
    max_words: int,
    ideal_words: int,
) -> tuple[int, int, str]:
    lines = raw_text.splitlines(keepends=True)
    source_words = count_words(raw_text)

    if not lines:
        return 1, 1, "empty file"
    if source_words <= max_words:
        return 1, len(lines), "source already within cap"

    numbered_view, visible_line_count = build_numbered_view(lines, MAX_PROMPT_CHARS)
    if visible_line_count == 0:
        return 1, len(lines), "no visible lines available for selection"

    counts = prefix_word_counts(lines)
    attempt_note = ""
    last_reason = ""

    for _ in range(MAX_SELECTION_ATTEMPTS):
        prompt = build_prompt(
            mode,
            group_name,
            numbered_view,
            visible_line_count,
            len(lines),
            source_words,
            min_words,
            max_words,
            ideal_words,
            attempt_note,
        )
        result = call_model_json(prompt)
        start_line = max(1, min(int(result.get("start_line", 1)), visible_line_count))
        end_line = max(
            start_line, min(int(result.get("end_line", start_line)), visible_line_count)
        )
        last_reason = str(result.get("reason", "")).strip()
        span_words = words_between(counts, start_line, end_line)

        if min_words <= span_words <= max_words:
            return start_line, end_line, last_reason

        direction = "shorter" if span_words > max_words else "longer"
        attempt_note = (
            f"Previous selection was approximately {span_words} words. "
            f"Choose a {direction} contiguous range within the visible lines."
        )

    start_line, end_line = find_best_fallback_window(
        lines[:visible_line_count], min_words, max_words, ideal_words
    )
    return (
        start_line,
        end_line,
        f"fallback_window_used; last_model_reason={last_reason}",
    )


records = []

for group_name, info in SOURCE_GROUPS.items():
    source_root = info["src"]
    destination_root = info["dst"]
    target = TARGETS[group_name]
    text_files = sorted(path for path in source_root.rglob("*.txt") if path.is_file())

    print(f'\n=== {group_name} | mode={info["mode"]} | files={len(text_files)} ===')

    for index, source_path in enumerate(text_files, start=1):
        relative_path = source_path.relative_to(source_root)
        destination_path = destination_root / relative_path
        destination_path.parent.mkdir(parents=True, exist_ok=True)

        if destination_path.exists() and not OVERWRITE_EXISTING:
            print(f"[{index}/{len(text_files)}] SKIP existing: {relative_path}")
            records.append(
                {
                    "group": group_name,
                    "file": str(relative_path),
                    "status": "skipped_existing",
                    "source_words": None,
                    "output_words": None,
                    "start_line": None,
                    "end_line": None,
                    "reason": "",
                    "error": "",
                }
            )
            continue

        print(f"[{index}/{len(text_files)}] Processing: {relative_path}")

        try:
            raw_text = read_raw_text(source_path)
            source_words = count_words(raw_text)
            lines = raw_text.splitlines(keepends=True)

            if not lines:
                kept_text, start_line, end_line, reason = "", 1, 1, "empty file"
            else:
                start_line, end_line, reason = choose_span_with_model(
                    raw_text,
                    info["mode"],
                    group_name,
                    target["min_words"],
                    target["max_words"],
                    target["ideal_words"],
                )
                kept_text, start_line, end_line = extract_exact_span(
                    raw_text, start_line, end_line
                )

            output_words = count_words(kept_text)
            write_raw_text(destination_path, kept_text)
            print(
                f"    lines={start_line}-{end_line} | "
                f"source_words={source_words} | output_words={output_words}"
            )

            records.append(
                {
                    "group": group_name,
                    "file": str(relative_path),
                    "status": "success",
                    "source_words": source_words,
                    "output_words": output_words,
                    "start_line": start_line,
                    "end_line": end_line,
                    "reason": reason,
                    "error": "",
                }
            )
        except Exception as error:
            print(f"    FAILED: {error}")
            records.append(
                {
                    "group": group_name,
                    "file": str(relative_path),
                    "status": "failed",
                    "source_words": None,
                    "output_words": None,
                    "start_line": None,
                    "end_line": None,
                    "reason": "",
                    "error": str(error),
                }
            )

log_df = pd.DataFrame(records)
log_path = OUTPUT_BASE_DIR / "_exact_span_trim_log.csv"
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

summary_rows = []
for group_name in SOURCE_GROUPS:
    successful = log_df[
        (log_df["group"] == group_name) & (log_df["status"] == "success")
    ]
    summary_rows.append(
        {
            "group": group_name,
            "files_success": int(len(successful)),
            "source_total_words": (
                int(successful["source_words"].fillna(0).sum())
                if len(successful)
                else 0
            ),
            "output_total_words": (
                int(successful["output_words"].fillna(0).sum())
                if len(successful)
                else 0
            ),
            "avg_output_words": (
                round(float(successful["output_words"].mean()), 2)
                if len(successful)
                else 0.0
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_BASE_DIR / "_exact_span_trim_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nDONE")
print(summary_df.to_string(index=False))
print(f"\nOutput root: {OUTPUT_BASE_DIR}")
print(f"Log CSV: {log_path}")
print(f"Summary CSV: {summary_path}")

In [ ]:
import html
import os
import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

# Set this only when the data directory is accessed through Google Drive in Colab.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

project_data_path = os.environ.get("PROJECT_DATA_PATH")
if not project_data_path:
    raise EnvironmentError(
        "Set the PROJECT_DATA_PATH environment variable before running this cell."
    )

BASE_TESTING_SAMPLE_DIR = Path(project_data_path).expanduser()
ORIGINAL_DIR = (
    BASE_TESTING_SAMPLE_DIR / "Arabic Assignments" / "Trimmed AI-Free Assignments"
)
TRIMMED_DIR = (
    BASE_TESTING_SAMPLE_DIR
    / "Exact_Span_Trimmed_For_AI_Detection"
    / "Arabic Assignments"
    / "Trimmed AI-Free Assignments"
)
OUTPUT_HTML_DIR = (
    BASE_TESTING_SAMPLE_DIR / "Arabic Assignments" / "Exact_Span_Trim_Review_HTML_10"
)

FILES_TO_PROCESS = 10
OVERWRITE_EXISTING = True

OUTPUT_HTML_DIR.mkdir(parents=True, exist_ok=True)


def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8-sig", errors="ignore")


def normalize_for_matching(text: str) -> str:
    """Reduce whitespace-only differences before token-level comparison."""
    text = text.replace("\u00a0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def tokenize_with_whitespace(text: str) -> list[str]:
    return re.findall(r"\S+|\s+", text, flags=re.UNICODE)


def tokens_to_html(tokens: list[str], css_class: str | None = None) -> str:
    content = html.escape("".join(tokens))
    return (
        content if css_class is None else f'<span class="{css_class}">{content}</span>'
    )


def build_highlight_html(original_text: str, trimmed_text: str) -> str:
    """Highlight original tokens retained in or removed by the trimmed version."""
    original_tokens = tokenize_with_whitespace(normalize_for_matching(original_text))
    trimmed_tokens = tokenize_with_whitespace(normalize_for_matching(trimmed_text))

    matcher = SequenceMatcher(a=original_tokens, b=trimmed_tokens, autojunk=False)
    parts = []

    for tag, i1, i2, _, _ in matcher.get_opcodes():
        original_chunk = original_tokens[i1:i2]

        if tag == "equal":
            parts.append(tokens_to_html(original_chunk, "kept"))
        elif tag in {"delete", "replace"} and original_chunk:
            parts.append(tokens_to_html(original_chunk, "removed"))

    return "".join(parts)


def build_full_html(
    file_name: str,
    rel_path: str,
    original_words: int,
    trimmed_words: int,
    body_html: str,
) -> str:
    reduction = original_words - trimmed_words
    reduction_pct = 100.0 * reduction / original_words if original_words else 0.0

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{html.escape(file_name)} - Exact Span Trim Review</title>
<style>
    body {{
        font-family: Arial, Helvetica, sans-serif;
        margin: 24px;
        background: #ffffff;
        color: #111111;
    }}
    .meta {{
        border: 1px solid #d9d9d9;
        border-radius: 10px;
        padding: 16px 18px;
        margin-bottom: 20px;
        background: #fafafa;
    }}
    .meta h1 {{
        margin: 0 0 10px;
        font-size: 20px;
    }}
    .meta p {{
        margin: 6px 0;
        line-height: 1.45;
    }}
    .legend {{
        margin-top: 12px;
    }}
    .legend span {{
        display: inline-block;
        padding: 4px 8px;
        margin-right: 10px;
        border: 1px solid #cccccc;
        border-radius: 6px;
    }}
    .doc {{
        border: 1px solid #d9d9d9;
        border-radius: 10px;
        padding: 20px;
        white-space: pre-wrap;
        line-height: 1.9;
        font-size: 16px;
        direction: rtl;
        text-align: right;
    }}
    .kept {{
        background-color: #c6efce;
        color: #1f4e2a;
    }}
    .removed {{
        background-color: #ffc7ce;
        color: #8a1f2d;
    }}
</style>
</head>
<body>
    <div class="meta">
        <h1>{html.escape(file_name)}</h1>
        <p><strong>Relative path:</strong> {html.escape(rel_path)}</p>
        <p><strong>Original words:</strong> {original_words}</p>
        <p><strong>Trimmed words:</strong> {trimmed_words}</p>
        <p><strong>Words removed:</strong> {reduction} ({reduction_pct:.2f}%)</p>
        <div class="legend">
            <span class="kept">Kept in trimmed file</span>
            <span class="removed">Removed from original file</span>
        </div>
    </div>
    <div class="doc">{body_html}</div>
</body>
</html>"""


def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text, flags=re.UNICODE))


if not ORIGINAL_DIR.exists():
    raise FileNotFoundError(f"Original folder not found:\n{ORIGINAL_DIR}")

if not TRIMMED_DIR.exists():
    raise FileNotFoundError(f"Trimmed folder not found:\n{TRIMMED_DIR}")

original_files = sorted(path for path in ORIGINAL_DIR.rglob("*.txt") if path.is_file())
selected_files = original_files[:FILES_TO_PROCESS]

if len(selected_files) < FILES_TO_PROCESS:
    print(f"Warning: only found {len(selected_files)} files.")

records = []
print(f"Selected {len(selected_files)} files.")

for index, original_path in enumerate(selected_files, start=1):
    relative_path = original_path.relative_to(ORIGINAL_DIR)
    trimmed_path = TRIMMED_DIR / relative_path
    html_path = OUTPUT_HTML_DIR / relative_path.with_suffix(".html")
    html_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"[{index}/{len(selected_files)}] {relative_path.name}")

    if html_path.exists() and not OVERWRITE_EXISTING:
        records.append(
            {
                "file": str(relative_path),
                "status": "skipped_existing_html",
                "original_exists": True,
                "trimmed_exists": trimmed_path.exists(),
                "original_words": None,
                "trimmed_words": None,
                "html_path": str(html_path),
                "error": "",
            }
        )
        continue

    if not trimmed_path.exists():
        print("    Missing trimmed version")
        records.append(
            {
                "file": str(relative_path),
                "status": "missing_trimmed_file",
                "original_exists": True,
                "trimmed_exists": False,
                "original_words": None,
                "trimmed_words": None,
                "html_path": "",
                "error": "Trimmed file not found.",
            }
        )
        continue

    try:
        original_text = read_text(original_path)
        trimmed_text = read_text(trimmed_path)
        original_words = count_words(original_text)
        trimmed_words = count_words(trimmed_text)

        full_html = build_full_html(
            file_name=original_path.name,
            rel_path=str(relative_path),
            original_words=original_words,
            trimmed_words=trimmed_words,
            body_html=build_highlight_html(original_text, trimmed_text),
        )
        html_path.write_text(full_html, encoding="utf-8")

        records.append(
            {
                "file": str(relative_path),
                "status": "success",
                "original_exists": True,
                "trimmed_exists": True,
                "original_words": original_words,
                "trimmed_words": trimmed_words,
                "html_path": str(html_path),
                "error": "",
            }
        )
        print(f"    Saved HTML: {html_path.name}")

    except Exception as exc:
        print(f"    FAILED: {exc}")
        records.append(
            {
                "file": str(relative_path),
                "status": "failed",
                "original_exists": True,
                "trimmed_exists": True,
                "original_words": None,
                "trimmed_words": None,
                "html_path": "",
                "error": str(exc),
            }
        )

log_df = pd.DataFrame(records)
log_path = OUTPUT_HTML_DIR / "_exact_span_arabic_html_review_log_10.csv"
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

print(f"\nHTML review folder:\n{OUTPUT_HTML_DIR}")
print(f"\nLog file:\n{log_path}")
print("\nStatus counts:")
print(log_df["status"].value_counts(dropna=False).to_string())

In [ ]:
# Dependencies: google-cloud-documentai, openai, pymupdf, pandas
import json
import os
import re
import shutil
import time
from pathlib import Path

import fitz
import pandas as pd
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
from openai import OpenAI

# Configure these environment variables before running:
# INPUT_PDF_DIR, FINAL_OUTPUT_DIR, TEMP_BASE_DIR,
# GOOGLE_SERVICE_ACCOUNT_JSON, GCP_PROJECT_ID, GCP_LOCATION,
# DOC_AI_PROCESSOR_ID, OPENAI_API_KEY
INPUT_PDF_DIR = os.environ["INPUT_PDF_DIR"]
FINAL_OUTPUT_DIR = os.environ["FINAL_OUTPUT_DIR"]
TEMP_BASE_DIR = os.environ["TEMP_BASE_DIR"]
GOOGLE_SERVICE_ACCOUNT_JSON = os.environ["GOOGLE_SERVICE_ACCOUNT_JSON"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us")
DOC_AI_PROCESSOR_ID = os.environ["DOC_AI_PROCESSOR_ID"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

OCR_CLEAN_MODEL = "gpt-5.4-mini"
EXACT_SPAN_MODEL = "gpt-4.1-mini"
DOC_AI_MAX_PAGES_PER_CHUNK = 15
OPENAI_CLEAN_CHUNK_WORDS = 3500
OCR_FINAL_WORD_CAP = 6000
SPAN_MIN_WORDS = 1200
SPAN_MAX_WORDS = 2000
SPAN_IDEAL_WORDS = 1500
OVERWRITE_EXISTING_FINALS = False
MAX_RETRIES_API = 4
MAX_SELECTION_ATTEMPTS = 3
SLEEP_BETWEEN_OPENAI_CALLS = 1.0
SLEEP_BETWEEN_DOC_AI_CALLS = 0.4
MAX_PROMPT_CHARS = 120000

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GOOGLE_SERVICE_ACCOUNT_JSON

input_pdf_dir = Path(INPUT_PDF_DIR)
final_output_dir = Path(FINAL_OUTPUT_DIR)
temp_base_dir = Path(TEMP_BASE_DIR)

if not input_pdf_dir.is_dir():
    raise FileNotFoundError(f"Input PDF folder not found: {input_pdf_dir}")

for directory in [
    final_output_dir,
    temp_base_dir,
    temp_base_dir / "_raw_ocr",
    temp_base_dir / "_cleaned_full",
    temp_base_dir / "_ocr_trimmed",
    temp_base_dir / "_tmp_pdf_chunks",
]:
    directory.mkdir(parents=True, exist_ok=True)

raw_ocr_dir = temp_base_dir / "_raw_ocr"
clean_full_dir = temp_base_dir / "_cleaned_full"
ocr_trimmed_dir = temp_base_dir / "_ocr_trimmed"
tmp_pdf_chunk_dir = temp_base_dir / "_tmp_pdf_chunks"

docai_client = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(
        api_endpoint=f"{GCP_LOCATION}-documentai.googleapis.com"
    )
)
processor_name = docai_client.processor_path(
    GCP_PROJECT_ID, GCP_LOCATION, DOC_AI_PROCESSOR_ID
)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

ARABIC_LETTERS_RE = re.compile(r"[\u0600-\u06FF]")


def count_words(text):
    return len(re.findall(r"\S+", text or "", flags=re.UNICODE))


def normalize_spaces(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+|[ \t]+\n", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def has_arabic(text):
    return bool(ARABIC_LETTERS_RE.search(text or ""))


def text_anchor_to_str(document_text, text_anchor):
    if not text_anchor or not text_anchor.text_segments:
        return ""
    return "".join(
        document_text[int(segment.start_index or 0) : int(segment.end_index or 0)]
        for segment in text_anchor.text_segments
    )


def extract_structured_text_from_docai(document):
    full_text = document.text or ""
    pages = []
    for page in getattr(document, "pages", []):
        paragraphs = [
            normalize_spaces(
                text_anchor_to_str(full_text, paragraph.layout.text_anchor)
            )
            for paragraph in getattr(page, "paragraphs", [])
        ]
        paragraphs = [paragraph for paragraph in paragraphs if paragraph]
        if not paragraphs:
            paragraphs = [
                normalize_spaces(text_anchor_to_str(full_text, line.layout.text_anchor))
                for line in getattr(page, "lines", [])
            ]
            paragraphs = [paragraph for paragraph in paragraphs if paragraph]
        if paragraphs:
            pages.append("\n\n".join(paragraphs))
    return normalize_spaces("\n\n".join(pages) or full_text)


def split_pdf_into_chunks(pdf_path, out_dir, max_pages=15):
    document = fitz.open(pdf_path)
    chunk_paths = []
    try:
        for start in range(0, document.page_count, max_pages):
            end = min(start + max_pages, document.page_count)
            chunk_path = (
                out_dir / f"{pdf_path.stem}__pages_{start + 1:03d}_{end:03d}.pdf"
            )
            chunk = fitz.open()
            try:
                chunk.insert_pdf(document, from_page=start, to_page=end - 1)
                chunk.save(chunk_path)
            finally:
                chunk.close()
            chunk_paths.append(chunk_path)
        return chunk_paths, document.page_count
    finally:
        document.close()


def ocr_pdf_chunk_with_docai(pdf_chunk_path):
    raw_document = documentai.RawDocument(
        content=pdf_chunk_path.read_bytes(), mime_type="application/pdf"
    )
    result = docai_client.process_document(
        request=documentai.ProcessRequest(
            name=processor_name, raw_document=raw_document
        )
    )
    return normalize_spaces(extract_structured_text_from_docai(result.document))


def basic_post_ocr_cleanup(text):
    cleaned, previous = [], None
    for line in (line.strip() for line in text.splitlines()):
        if not line:
            cleaned.append("")
        elif re.fullmatch(r"[\d\W_]+", line):
            continue
        elif line == previous and len(line) <= 40:
            continue
        else:
            cleaned.append(line)
        previous = line
    return normalize_spaces("\n".join(cleaned))


def split_text_paragraphwise(text, max_words=3500):
    paragraphs = [
        p.strip() for p in re.split(r"\n\s*\n", normalize_spaces(text)) if p.strip()
    ]
    chunks, current, current_words = [], [], 0
    for paragraph in paragraphs:
        words = count_words(paragraph)
        if current and current_words + words > max_words:
            chunks.append("\n\n".join(current))
            current, current_words = [paragraph], words
        else:
            current.append(paragraph)
            current_words += words
    if current:
        chunks.append("\n\n".join(current))

    result = []
    for chunk in chunks:
        if count_words(chunk) <= max_words:
            result.append(chunk)
            continue
        current, words = [], 0
        for token in re.findall(r"\S+|\s+", chunk, flags=re.UNICODE):
            if not token.isspace() and words == max_words:
                result.append("".join(current).strip())
                current, words = [], 0
            current.append(token)
            if not token.isspace():
                words += 1
        if current:
            result.append("".join(current).strip())
    return [chunk for chunk in result if chunk]


def call_openai_text(model_name, prompt):
    last_error = None
    for attempt in range(1, MAX_RETRIES_API + 1):
        try:
            output = (
                openai_client.responses.create(
                    model=model_name, input=prompt
                ).output_text
                or ""
            ).strip()
            if not output:
                raise ValueError("Empty model output.")
            time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
            return output
        except Exception as error:
            last_error = error
            if attempt == MAX_RETRIES_API:
                raise
            time.sleep(min(2 ** (attempt - 1), 12))
    raise last_error


def openai_clean_ocr_chunk(text):
    prompt = f"""Clean OCR-extracted Arabic academic text while preserving meaning and order.
Correct only clear OCR errors. Join broken paragraph lines. Remove repeated headers, footers,
page numbers, garbage symbols, duplicate fragments, and scanning artifacts. Preserve headings,
bullets, and paragraphs. Do not summarize, translate, paraphrase, or add commentary.
Return Arabic text only.

TEXT:
{text}"""
    return normalize_spaces(call_openai_text(OCR_CLEAN_MODEL, prompt))


def openai_trim_to_cap(text, word_cap=6000):
    prompt = f"""Select coherent Arabic academic prose for AI-detection input.
Remove title pages, author metadata, contents, repeated headers and footers, page numbers,
references, bibliography, appendices, URLs, isolated equations, and figure or table captions.
Preserve original order, complete paragraphs, and meaningful headings or bullets. Do not summarize
or paraphrase retained content. Return at most {word_cap} words and no commentary.

TEXT:
{text}"""
    return normalize_spaces(call_openai_text(OCR_CLEAN_MODEL, prompt))


def hard_cap_to_full_paragraphs(text, word_cap=6000):
    paragraphs = [
        p.strip() for p in re.split(r"\n\s*\n", normalize_spaces(text)) if p.strip()
    ]
    kept, total = [], 0
    for paragraph in paragraphs:
        words = count_words(paragraph)
        if kept and total + words > word_cap:
            break
        if not kept and words > word_cap:
            return normalize_spaces(" ".join(re.findall(r"\S+", paragraph)[:word_cap]))
        kept.append(paragraph)
        total += words
    return normalize_spaces("\n\n".join(kept))


def robust_json_extract(text):
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Could not parse JSON from model output: {text[:1000]}")


def call_model_json(prompt):
    for attempt in range(1, MAX_RETRIES_API + 1):
        try:
            response = openai_client.chat.completions.create(
                model=EXACT_SPAN_MODEL,
                response_format={"type": "json_object"},
                temperature=0,
                messages=[
                    {
                        "role": "developer",
                        "content": "Return valid JSON only with exactly: start_line, end_line, reason.",
                    },
                    {"role": "user", "content": prompt},
                ],
            )
            output = response.choices[0].message.content
            if not output:
                raise ValueError("Empty model output.")
            time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)
            return robust_json_extract(output)
        except Exception:
            if attempt == MAX_RETRIES_API:
                raise
            time.sleep(min(2 ** (attempt - 1), 12))


def find_best_fallback_window(lines, min_words, max_words, ideal_words):
    counts = [count_words(line) for line in lines]
    prefix = [sum(counts[: index + 1]) for index in range(len(counts))]
    best = None
    for start in range(len(lines)):
        for end in range(start, len(lines)):
            words = prefix[end] - (prefix[start - 1] if start else 0)
            score = (
                0 if min_words <= words <= max_words else 1,
                abs(words - ideal_words),
            )
            if best is None or score < best[0]:
                best = (score, start + 1, end + 1)
            if words > max_words:
                break
    return (best[1], best[2]) if best else (1, 1)


def choose_span_with_model(raw_text, min_words, max_words, ideal_words):
    lines = raw_text.splitlines(keepends=True)
    if not lines:
        return 1, 1, "empty file"
    if count_words(raw_text) <= max_words:
        return 1, len(lines), "source already within cap"

    view, visible = [], 0
    chars = 0
    for index, line in enumerate(lines, 1):
        row = f"{index:05d} | cw={count_words(line):03d} | {line.rstrip()}\n"
        if chars + len(row) > MAX_PROMPT_CHARS:
            break
        view.append(row)
        chars += len(row)
        visible = index

    note = ""
    for _ in range(MAX_SELECTION_ATTEMPTS):
        prompt = f"""Choose one contiguous Arabic academic-text excerpt for AI detection.
Prefer substantive prose and complete paragraphs; avoid metadata, contents, references,
headers, footers, and clutter. Select lines totaling {min_words}-{max_words} words,
ideally {ideal_words}. Only lines 1-{visible} are available.
Return JSON: {{\"start_line\": 1, \"end_line\": 2, \"reason\": \"...\"}}.
{note}

{''.join(view)}"""
        data = call_model_json(prompt)
        start = max(1, min(int(data.get("start_line", 1)), visible))
        end = max(start, min(int(data.get("end_line", start)), visible))
        words = count_words("".join(lines[start - 1 : end]))
        if min_words <= words <= max_words:
            return start, end, str(data.get("reason", ""))
        note = f"Previous selection had about {words} words; select a {'shorter' if words > max_words else 'longer'} range."

    start, end = find_best_fallback_window(
        lines[:visible], min_words, max_words, ideal_words
    )
    return start, end, "fallback_window_used"


pdf_files = sorted(path for path in input_pdf_dir.rglob("*.pdf") if path.is_file())
txt_files = sorted(path for path in final_output_dir.rglob("*.txt") if path.is_file())
pdf_stems = {path.stem: path for path in pdf_files}
txt_stems = {path.stem: path for path in txt_files}
missing_stems = sorted(set(pdf_stems) - set(txt_stems))

print(f"PDF files: {len(pdf_files)}")
print(f"Existing TXT files: {len(txt_files)}")
print(f"Missing files: {len(missing_stems)}")

records = []
for index, stem in enumerate(missing_stems, 1):
    pdf_path = pdf_stems[stem]
    final_out = final_output_dir / f"{stem}.txt"
    row = {
        "file": pdf_path.name,
        "stem": stem,
        "status": "",
        "pdf_pages": None,
        "raw_ocr_words": None,
        "clean_words": None,
        "ocr_trimmed_words": None,
        "exact_span_words": None,
        "start_line": None,
        "end_line": None,
        "error": "",
    }
    chunk_dir = tmp_pdf_chunk_dir / stem

    try:
        if final_out.exists() and not OVERWRITE_EXISTING_FINALS:
            row["status"] = "skipped_existing_final"
            continue
        if chunk_dir.exists():
            shutil.rmtree(chunk_dir)
        chunk_dir.mkdir(parents=True)

        chunks, row["pdf_pages"] = split_pdf_into_chunks(
            pdf_path, chunk_dir, DOC_AI_MAX_PAGES_PER_CHUNK
        )
        raw_text = basic_post_ocr_cleanup(
            "\n\n".join(ocr_pdf_chunk_with_docai(chunk) for chunk in chunks)
        )
        if not raw_text or not has_arabic(raw_text):
            raise RuntimeError("OCR returned no Arabic text.")
        (raw_ocr_dir / f"{stem}.txt").write_text(raw_text, encoding="utf-8")
        row["raw_ocr_words"] = count_words(raw_text)

        cleaned_text = normalize_spaces(
            "\n\n".join(
                openai_clean_ocr_chunk(chunk)
                for chunk in split_text_paragraphwise(
                    raw_text, OPENAI_CLEAN_CHUNK_WORDS
                )
            )
        )
        if not cleaned_text:
            raise RuntimeError("OCR cleanup returned empty text.")
        (clean_full_dir / f"{stem}.txt").write_text(cleaned_text, encoding="utf-8")
        row["clean_words"] = count_words(cleaned_text)

        trimmed_text = hard_cap_to_full_paragraphs(
            openai_trim_to_cap(cleaned_text, OCR_FINAL_WORD_CAP), OCR_FINAL_WORD_CAP
        )
        if not trimmed_text or not has_arabic(trimmed_text):
            raise RuntimeError("OCR trimming returned no Arabic text.")
        (ocr_trimmed_dir / f"{stem}.txt").write_text(trimmed_text, encoding="utf-8")
        row["ocr_trimmed_words"] = count_words(trimmed_text)

        start, end, _ = choose_span_with_model(
            trimmed_text, SPAN_MIN_WORDS, SPAN_MAX_WORDS, SPAN_IDEAL_WORDS
        )
        exact_span = "".join(trimmed_text.splitlines(keepends=True)[start - 1 : end])
        if not exact_span.strip():
            raise RuntimeError("Exact-span output is empty.")
        final_out.write_text(exact_span, encoding="utf-8")
        row.update(
            status="success",
            exact_span_words=count_words(exact_span),
            start_line=start,
            end_line=end,
        )
    except Exception as error:
        row.update(status="failed", error=str(error))
    finally:
        shutil.rmtree(chunk_dir, ignore_errors=True)
        records.append(row)

log_path = temp_base_dir / "_missing_recovery_log.csv"
pd.DataFrame(records).to_csv(log_path, index=False, encoding="utf-8-sig")
print(f"Log saved to: {log_path}")
print(f"Final TXT files: {len(list(final_output_dir.rglob('*.txt')))}")